In [56]:

import os
import warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
import optuna
import ast
import re
from best_params import xgboost_params, lightgbm_params
# Function to compute MSE scores
def compute_mse_scores(X_train, X_test, y_train, y_test, features):
    if not features:  # If no features are provided, return a high MSE (or a default value)
        return {'XGBoost': np.inf, 'LightGBM': np.inf}, np.inf, {}

    X_train = X_train[features].dropna()
    X_test = X_test[features].dropna()

    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse, {}
# Paths for processed data and TSFRESH features
processed_train_path = 'data/processed/train_transformed_combined.csv'
processed_test_path = 'data/processed/test_transformed_combined.csv'
tsfresh_train_path = 'data/tsfresh/train_combined_all_features_filled.csv'
tsfresh_test_path = 'data/tsfresh/test_combined_all_features_filled.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)

# Targets to evaluate
targets = ['FEDFUNDS', 'GDP', 'CPIAUCSL', 'CUSR0000SAH1', 'CPILFESL', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'HOUST', 'DSPI', 
           'DGS2', 'DGS5', 'DGS10', 'AAA', 'BAA', 'WTISPLC', 'IMPGS', 'GCE', 'FGCE', 'GDPCTPI', 'PCEPI', 'PCEPILFE', 
           'PAYEMS', 'UNRATE', 'INDPRO', 'CUMFNS', 'USREC']

# Model parameters
xgboost_params = {'max_depth': 9, 'learning_rate': 0.0168, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5866, 'colsample_bytree': 0.9177, 'reg_alpha': 0.0125, 'reg_lambda': 8.577e-05, 'verbosity': 0}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402, 'colsample_bytree': 0.9405, 'reg_alpha': 0.000756, 'reg_lambda': 0.000259, 'verbosity': -1}

# Path to results file
feature_engineering_results = pd.read_csv(r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\feature_engineering_results.csv')


In [3]:
# Function to compute MSE scores
def compute_mse_scores(X_train, X_test, y_train, y_test, features):
    if not features:  # If no features are provided, return a high MSE (or a default value)
        return {'XGBoost': np.inf, 'LightGBM': np.inf}, np.inf, {}

    X_train = X_train[features].dropna()
    X_test = X_test[features].dropna()

    # scaler = StandardScaler()
    X_train_scaled = X_train
    X_test_scaled = X_train

    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(**lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        mse_scores[model_name].append(mean_squared_error(y_test, y_pred))

    mse_scores = {model: np.mean(scores) for model, scores in mse_scores.items()}
    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse, {}

In [4]:
import ast
import re

target = 'FEDFUNDS'
print(f"Target: {target}")
# Extract base features for the target
base_features_str = feature_engineering_results.loc[feature_engineering_results['target'] == target]['final_feature_space'].values[0]

# Convert the string representation of the list to an actual list
base_features = ast.literal_eval(base_features_str)

# Define a function to replace ""text"" with "text"
def clean_double_quotes(feature_name):
    return re.sub(r'""(.*?)""', r'"\1"', feature_name)

# Apply the cleaning function to all feature names
base_features = [clean_double_quotes(feature) for feature in base_features]

# Verify that these features exist in the tsfresh data
missing_features = [feature for feature in base_features if feature not in tsfresh_features_train.columns]
if missing_features:
    print(f"Warning: The following features are missing from the tsfresh data: {missing_features}")

# Proceed with the features that are valid
valid_features = [feature for feature in base_features if feature in tsfresh_features_train.columns]

# Now use the cleaned and verified base_features list
X_train = tsfresh_features_train[valid_features]  # Select columns matching these features
X_test = tsfresh_features_test[valid_features]  # Select columns matching these features
y_train = train_combined[[target]]
y_test = test_combined[[target]]

# Continue with MSE calculation and the rest of your logic
baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, valid_features)

#### PRINT
print("1. BASE FEATURE EVAL:")
print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

# Append CUMFNS__has_duplicate, to the base features and test gain
base_features.append('CUMFNS__has_duplicate')
X_train = tsfresh_features_train[base_features]
X_test = tsfresh_features_test[base_features]
new_mse_scores, new_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, base_features)
print("2. ADDING CUMFNS__has_duplicate:")
print(f"Aggregated MSE: {new_aggregated_mse}")
print(f"IMPROVEMENT: {aggregated_baseline_mse - new_aggregated_mse}")
print(f"MSE for XGBoost: {new_mse_scores['XGBoost']}")
# % improvement
print(F" PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")

print("---------------------------------")

# Append  FGCE__has_duplicate_min to the base features and test gain
base_features.append('FGCE__has_duplicate_min')
X_train = tsfresh_features_train[base_features]
X_test = tsfresh_features_test[base_features]
new_mse_scores, new_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, base_features)
print("3. ADDING FGCE__has_duplicate_min:")
print(f"Aggregated MSE: {new_aggregated_mse}")
print(f"IMPROVEMENT: {aggregated_baseline_mse - new_aggregated_mse}")
print(f"MSE for XGBoost: {new_mse_scores['XGBoost']}")
# % improvement
print(F" PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")

# Append INDPRO__minimum to the base features and test gain
base_features.append('INDPRO__minimum')
X_train = tsfresh_features_train[base_features]
X_test = tsfresh_features_test[base_features]
new_mse_scores, new_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, base_features)
print("4. ADDING INDPRO__minimum:")
print(f"Aggregated MSE: {new_aggregated_mse}")
print(f"IMPROVEMENT: {aggregated_baseline_mse - new_aggregated_mse}")
print(f"MSE for XGBoost: {new_mse_scores['XGBoost']}")
# % improvement
print(F" PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")
print("---------------------------------")
print("4. ADDING FEDFUNDS__has_duplicate:")
base_features.append('FEDFUNDS__has_duplicate')
X_train = tsfresh_features_train[base_features]
X_test = tsfresh_features_test[base_features]
new_mse_scores, new_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, base_features)
print(f"Aggregated MSE: {new_aggregated_mse}")
print(f"IMPROVEMENT: {aggregated_baseline_mse - new_aggregated_mse}")
print(f"MSE for XGBoost: {new_mse_scores['XGBoost']}")
# % improvement
print(F" PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")
print("---------------------------------")
print("5. ADDING FEDFUNDS__minimum")
base_features.append('FEDFUNDS__minimum')
X_train = tsfresh_features_train[base_features]
X_test = tsfresh_features_test[base_features]
new_mse_scores, new_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, base_features)
print(f"Aggregated MSE: {new_aggregated_mse}")
print(f"IMPROVEMENT: {aggregated_baseline_mse - new_aggregated_mse}")
print(f"MSE for XGBoost: {new_mse_scores['XGBoost']}")
# % improvement
print(F" PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")
print("---------------------------------")
print('6. ADDING BAA__large_standard_deviation__r_0.2')
base_features.append('BAA__large_standard_deviation__r_0.2')
X_train = tsfresh_features_train[base_features]
X_test = tsfresh_features_test[base_features]
new_mse_scores, new_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, base_features)
print(f"Aggregated MSE: {new_aggregated_mse}")
print(f"IMPROVEMENT: {aggregated_baseline_mse - new_aggregated_mse}")
print(f"MSE for XGBoost: {new_mse_scores['XGBoost']}")
print(F" PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")
print("---------------------------------")


Target: FEDFUNDS


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.013832248973208266
Initial MSE for XGBoost: 0.007369381570450693
Initial MSE for LightGBM: 0.006462867402757572


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


2. ADDING CUMFNS__has_duplicate:
Aggregated MSE: 0.012976108576945911
IMPROVEMENT: 0.0008561403962623546
MSE for XGBoost: 0.00651324117418834
 PERCENT IMPROVEMENT: 6.1894518954988165%
---------------------------------


...

Aggregated MSE: 0.013763187776516018
IMPROVEMENT: 6.906119669224803e-05
MSE for XGBoost: 0.007131107003277486
 PERCENT IMPROVEMENT: 0.4992767034920563%
---------------------------------
5. ADDING FEDFUNDS__minimum


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Aggregated MSE: 0.01469250057642682
IMPROVEMENT: -0.0008602516032185543
MSE for XGBoost: 0.00806041980318829
 PERCENT IMPROVEMENT: -6.219173793681555%
---------------------------------
6. ADDING BAA__large_standard_deviation__r_0.2


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Aggregated MSE: 0.0150947951449182
IMPROVEMENT: -0.0012625461717099343
MSE for XGBoost: 0.008731763226069502
 PERCENT IMPROVEMENT: -9.12755528154073%
---------------------------------


In [7]:
import ast
import re

target = 'GDP'
print(f"Target: {target}")
# Extract base features for the target
base_features_str = feature_engineering_results.loc[feature_engineering_results['target'] == target]['final_feature_space'].values[0]

# Convert the string representation of the list to an actual list
base_features = ast.literal_eval(base_features_str)

# Define a function to replace ""text"" with "text"
def clean_double_quotes(feature_name):
    return re.sub(r'""(.*?)""', r'"\1"', feature_name)

# Apply the cleaning function to all feature names
base_features = [clean_double_quotes(feature) for feature in base_features]

# Verify that these features exist in the tsfresh data
missing_features = [feature for feature in base_features if feature not in tsfresh_features_train.columns]
if missing_features:
    print(f"Warning: The following features are missing from the tsfresh data: {missing_features}")

# Proceed with the features that are valid
valid_features = [feature for feature in base_features if feature in tsfresh_features_train.columns]

# Now use the cleaned and verified base_features list
X_train = tsfresh_features_train[valid_features]  # Select columns matching these features
X_test = tsfresh_features_test[valid_features]  # Select columns matching these features
y_train = train_combined[[target]]
y_test = test_combined[[target]]

# Continue with MSE calculation and the rest of your logic
baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, valid_features)

#### PRINT
print("1. BASE FEATURE EVAL:")
print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

# Append WTISPLC__index_mass_quantile__q_0.4 to the base features and test gain
base_features.append('WTISPLC__index_mass_quantile__q_0.4')
X_train = tsfresh_features_train[base_features]
X_test = tsfresh_features_test[base_features]
new_mse_scores, new_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, base_features)
print("2. ADDING WTISPLC__index_mass_quantile__q_0.4:")
print(f"Aggregated MSE: {new_aggregated_mse}")
print(f"IMPROVEMENT: {aggregated_baseline_mse - new_aggregated_mse}")
print(f"MSE for XGBoost: {new_mse_scores['XGBoost']}")
print(F" PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")
print("---------------------------------")

# Append PNFI__index_mass_quantile__q_0.9 to the base features and test gain
base_features.append('PNFI__index_mass_quantile__q_0.9')
X_train = tsfresh_features_train[base_features]
X_test = tsfresh_features_test[base_features]
new_mse_scores, new_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, base_features)
print("3. ADDING PNFI__index_mass_quantile__q_0.9:")
print(f"Aggregated MSE: {new_aggregated_mse}")
print(f"IMPROVEMENT: {aggregated_baseline_mse - new_aggregated_mse}")
print(f"MSE for XGBoost: {new_mse_scores['XGBoost']}")
print(F" PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")
print("---------------------------------")


Target: GDP


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.0002489168743109566
Initial MSE for XGBoost: 0.00012544477858931284
Initial MSE for LightGBM: 0.00012347209572164373


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


2. ADDING WTISPLC__index_mass_quantile__q_0.4:
Aggregated MSE: 0.0002441687260137718
IMPROVEMENT: 4.74814829718481e-06
MSE for XGBoost: 0.00010820205540484314
 PERCENT IMPROVEMENT: 1.9075236704335439%
---------------------------------


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


3. ADDING PNFI__index_mass_quantile__q_0.9:
Aggregated MSE: 0.00023339910511365825
IMPROVEMENT: 1.5517769197298352e-05
MSE for XGBoost: 0.00012005422821848275
 PERCENT IMPROVEMENT: 6.234117008038979%
---------------------------------


In [9]:
import ast
import re

target = 'CPIAUCSL'
print(f"Target: {target}")
# Extract base features for the target
base_features_str = feature_engineering_results.loc[feature_engineering_results['target'] == target]['final_feature_space'].values[0]

# Convert the string representation of the list to an actual list
base_features = ast.literal_eval(base_features_str)

# Define a function to replace ""text"" with "text"
def clean_double_quotes(feature_name):
    return re.sub(r'""(.*?)""', r'"\1"', feature_name)

# Apply the cleaning function to all feature names
base_features = [clean_double_quotes(feature) for feature in base_features]

# Verify that these features exist in the tsfresh data
missing_features = [feature for feature in base_features if feature not in tsfresh_features_train.columns]
if missing_features:
    print(f"Warning: The following features are missing from the tsfresh data: {missing_features}")

# Proceed with the features that are valid
valid_features = [feature for feature in base_features if feature in tsfresh_features_train.columns]

# Now use the cleaned and verified base_features list
X_train = tsfresh_features_train[valid_features]  # Select columns matching these features
X_test = tsfresh_features_test[valid_features]  # Select columns matching these features
y_train = train_combined[[target]]
y_test = test_combined[[target]]

# Continue with MSE calculation and the rest of your logic
baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, valid_features)

#### PRINT
print("1. BASE FEATURE EVAL:")
print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

# Append features one by one and check the improvements
features_to_add = ['DGS10__longest_strike_below_mean','PRFI__maximum','CUMFNS__has_duplicate_min','BAA__sum_of_reoccurring_values','CPIAUCSL__large_standard_deviation__r_0.4']

for i, feature in enumerate(features_to_add, start=2):
    print(f"\n{i}. ADDING {feature}:")
    base_features.append(feature)
    X_train = tsfresh_features_train[base_features]
    X_test = tsfresh_features_test[base_features]
    new_mse_scores, new_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, base_features)
    print(f"Aggregated MSE: {new_aggregated_mse}")
    print(f"IMPROVEMENT: {aggregated_baseline_mse - new_aggregated_mse}")
    print(f"MSE for XGBoost: {new_mse_scores['XGBoost']}")
    print(f" PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")
    print("---------------------------------")


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 7.661181763559996e-05
Initial MSE for XGBoost: 3.800471050499076e-05
Initial MSE for LightGBM: 3.8607107130609205e-05

2. ADDING DGS10__longest_strike_below_mean:


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Aggregated MSE: 7.656330914277817e-05
IMPROVEMENT: 4.8508492821793415e-08
MSE for XGBoost: 3.795620201216897e-05
 PERCENT IMPROVEMENT: 0.06331724571856719%
---------------------------------

3. ADDING PRFI__maximum:


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


...

c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Aggregated MSE: 7.706483826062065e-05
IMPROVEMENT: -4.53020625020684e-07
MSE for XGBoost: 3.84549331314461e-05
 PERCENT IMPROVEMENT: -0.5913195104904736%
---------------------------------

5. ADDING BAA__sum_of_reoccurring_values:


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Aggregated MSE: 7.693411491498366e-05
IMPROVEMENT: -3.222972793837011e-07
MSE for XGBoost: 3.8324209785809114e-05
 PERCENT IMPROVEMENT: -0.420688725748149%
---------------------------------

6. ADDING CPIAUCSL__large_standard_deviation__r_0.4:
Aggregated MSE: 7.712183330891954e-05
IMPROVEMENT: -5.100156733195828e-07
MSE for XGBoost: 3.8511928179744995e-05
 PERCENT IMPROVEMENT: -0.6657141013746015%
---------------------------------


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [10]:
import ast
import re

target = 'CUSR0000SAH1'
print(f"Target: {target}")
# Extract base features for the target
base_features_str = feature_engineering_results.loc[feature_engineering_results['target'] == target]['final_feature_space'].values[0]

# Convert the string representation of the list to an actual list
base_features = ast.literal_eval(base_features_str)

# Define a function to replace ""text"" with "text"
def clean_double_quotes(feature_name):
    return re.sub(r'""(.*?)""', r'"\1"', feature_name)

# Apply the cleaning function to all feature names
base_features = [clean_double_quotes(feature) for feature in base_features]

# Verify that these features exist in the tsfresh data
missing_features = [feature for feature in base_features if feature not in tsfresh_features_train.columns]
if missing_features:
    print(f"Warning: The following features are missing from the tsfresh data: {missing_features}")

# Proceed with the features that are valid
valid_features = [feature for feature in base_features if feature in tsfresh_features_train.columns]

# Now use the cleaned and verified base_features list
X_train = tsfresh_features_train[valid_features]  # Select columns matching these features
X_test = tsfresh_features_test[valid_features]  # Select columns matching these features
y_train = train_combined[[target]]
y_test = test_combined[[target]]

# Continue with MSE calculation and the rest of your logic
baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, valid_features)

#### PRINT
print("1. BASE FEATURE EVAL:")
print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

# Feature addition steps
features_to_add = [
    'CUMFNS__has_duplicate_min', 
    'PCEPILFE__ratio_beyond_r_sigma__r_5', 
    'FEDFUNDS__has_duplicate_min', 
    'PAYEMS__symmetry_looking__r_0.1', 
    'WTISPLC__has_duplicate'
]

for i, feature in enumerate(features_to_add):
    base_features.append(feature)
    X_train = tsfresh_features_train[base_features]
    X_test = tsfresh_features_test[base_features]
    new_mse_scores, new_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, base_features)
    print(f"{i+2}. ADDING {feature}:")
    print(f"Aggregated MSE: {new_aggregated_mse}")
    print(f"IMPROVEMENT: {aggregated_baseline_mse - new_aggregated_mse}")
    print(f"MSE for XGBoost: {new_mse_scores['XGBoost']}")
    print(F" PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")
    print("---------------------------------")


Target: CUSR0000SAH1


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 8.290551312662667e-06
Initial MSE for XGBoost: 4.129408097213033e-06
Initial MSE for LightGBM: 4.161143215449635e-06


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


2. ADDING CUMFNS__has_duplicate_min:
Aggregated MSE: 8.29119661622559e-06
IMPROVEMENT: -6.453035629230131e-10
MSE for XGBoost: 4.130053400775955e-06
 PERCENT IMPROVEMENT: -0.007783602544470129%
---------------------------------


...

4. ADDING FEDFUNDS__has_duplicate_min:
Aggregated MSE: 8.289229187533482e-06
IMPROVEMENT: 1.3221251291852237e-09
MSE for XGBoost: 4.128085972083848e-06
 PERCENT IMPROVEMENT: 0.015947372850414192%
---------------------------------


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


5. ADDING PAYEMS__symmetry_looking__r_0.1:
Aggregated MSE: 8.281765407160403e-06
IMPROVEMENT: 8.785905502264104e-09
MSE for XGBoost: 4.120622191710767e-06
 PERCENT IMPROVEMENT: 0.10597492459693064%
---------------------------------


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


6. ADDING WTISPLC__has_duplicate:
Aggregated MSE: 8.287224940862077e-06
IMPROVEMENT: 3.3263718005906422e-09
MSE for XGBoost: 4.126081725412442e-06
 PERCENT IMPROVEMENT: 0.040122443914074456%
---------------------------------


In [11]:
import ast
import re

target = 'CPILFESL'
print(f"Target: {target}")
# Extract base features for the target
base_features_str = feature_engineering_results.loc[feature_engineering_results['target'] == target]['final_feature_space'].values[0]

# Convert the string representation of the list to an actual list
base_features = ast.literal_eval(base_features_str)

# Define a function to replace ""text"" with "text"
def clean_double_quotes(feature_name):
    return re.sub(r'""(.*?)""', r'"\1"', feature_name)

# Apply the cleaning function to all feature names
base_features = [clean_double_quotes(feature) for feature in base_features]

# Verify that these features exist in the tsfresh data
missing_features = [feature for feature in base_features if feature not in tsfresh_features_train.columns]
if missing_features:
    print(f"Warning: The following features are missing from the tsfresh data: {missing_features}")

# Proceed with the features that are valid
valid_features = [feature for feature in base_features if feature in tsfresh_features_train.columns]

# Now use the cleaned and verified base_features list
X_train = tsfresh_features_train[valid_features]  # Select columns matching these features
X_test = tsfresh_features_test[valid_features]  # Select columns matching these features
y_train = train_combined[[target]]
y_test = test_combined[[target]]

# Continue with MSE calculation and the rest of your logic
baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, valid_features)

#### PRINT
print("1. BASE FEATURE EVAL:")
print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

# Feature addition steps
features_to_add = [
    'DGS10__minimum', 
    'EXPGS__symmetry_looking__r_0.1', 
    'GDP__symmetry_looking__r_0.05', 
    'DGS2__large_standard_deviation__r_0.30000000000000004'
]

for i, feature in enumerate(features_to_add):
    base_features.append(feature)
    X_train = tsfresh_features_train[base_features]
    X_test = tsfresh_features_test[base_features]
    new_mse_scores, new_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, base_features)
    print(f"{i+2}. ADDING {feature}:")
    print(f"Aggregated MSE: {new_aggregated_mse}")
    print(f"IMPROVEMENT: {aggregated_baseline_mse - new_aggregated_mse}")
    print(f"MSE for XGBoost: {new_mse_scores['XGBoost']}")
    print(F" PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")
    print("---------------------------------")


Target: CPILFESL


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 1.956418719504441e-05
Initial MSE for XGBoost: 9.764956987185286e-06
Initial MSE for LightGBM: 9.799230207859125e-06


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


2. ADDING DGS10__minimum:
Aggregated MSE: 1.9565078193506845e-05
IMPROVEMENT: -8.909984624359672e-10
MSE for XGBoost: 9.765847985647718e-06
 PERCENT IMPROVEMENT: -0.0045542319420336375%
---------------------------------


...

3. ADDING EXPGS__symmetry_looking__r_0.1:
Aggregated MSE: 1.9570336250775312e-05
IMPROVEMENT: -6.149055730902611e-09
MSE for XGBoost: 9.771106042916186e-06
 PERCENT IMPROVEMENT: -0.03143016200775344%
---------------------------------


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


4. ADDING GDP__symmetry_looking__r_0.05:
Aggregated MSE: 1.9568447865752883e-05
IMPROVEMENT: -4.260670708473559e-09
MSE for XGBoost: 9.769217657893757e-06
 PERCENT IMPROVEMENT: -0.021777908103193692%
---------------------------------


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


5. ADDING DGS2__large_standard_deviation__r_0.30000000000000004:
Aggregated MSE: 1.9571814466417645e-05
IMPROVEMENT: -7.627271373235827e-09
MSE for XGBoost: 9.77258425855852e-06
 PERCENT IMPROVEMENT: -0.038985884244492446%
---------------------------------


In [13]:
import ast
import re

target = 'PCE'
print(f"Target: {target}")
# Extract base features for the target
base_features_str = feature_engineering_results.loc[feature_engineering_results['target'] == target]['final_feature_space'].values[0]

# Convert the string representation of the list to an actual list
base_features = ast.literal_eval(base_features_str)

# Define a function to replace ""text"" with "text"
def clean_double_quotes(feature_name):
    return re.sub(r'""(.*?)""', r'"\1"', feature_name)

# Apply the cleaning function to all feature names
base_features = [clean_double_quotes(feature) for feature in base_features]

# Verify that these features exist in the tsfresh data
missing_features = [feature for feature in base_features if feature not in tsfresh_features_train.columns]
if missing_features:
    print(f"Warning: The following features are missing from the tsfresh data: {missing_features}")

# Proceed with the features that are valid
valid_features = [feature for feature in base_features if feature in tsfresh_features_train.columns]

# Now use the cleaned and verified base_features list
X_train = tsfresh_features_train[valid_features]  # Select columns matching these features
X_test = tsfresh_features_test[valid_features]  # Select columns matching these features
y_train = train_combined[[target]]
y_test = test_combined[[target]]

# Continue with MSE calculation and the rest of your logic
baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, valid_features)

#### PRINT
print("1. BASE FEATURE EVAL:")
print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

# Feature addition steps
features_to_add = [
    'PCE__last_location_of_minimum', 
    'INDPRO__cid_ce__normalize_False', 
    'GDP__c3__lag_1', 
    'GDP__sum_values'
]

for i, feature in enumerate(features_to_add):
    base_features.append(feature)
    X_train = tsfresh_features_train[base_features]
    X_test = tsfresh_features_test[base_features]
    new_mse_scores, new_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, base_features)
    print(f"{i+2}. ADDING {feature}:")
    print(f"Aggregated MSE: {new_aggregated_mse}")
    print(f"IMPROVEMENT: {aggregated_baseline_mse - new_aggregated_mse}")
    print(f"MSE for XGBoost: {new_mse_scores['XGBoost']}")
    print(F" PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")
    print("---------------------------------")


Target: PCE


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.0007530484731545608
Initial MSE for XGBoost: 0.00035366810970258595
Initial MSE for LightGBM: 0.0003993803634519748


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


2. ADDING PCE__last_location_of_minimum:
Aggregated MSE: 0.0007283020010193722
IMPROVEMENT: 2.4746472135188553e-05
MSE for XGBoost: 0.00032620740184085266
 PERCENT IMPROVEMENT: 3.2861725396672337%
---------------------------------


...

3. ADDING INDPRO__cid_ce__normalize_False:
Aggregated MSE: 0.0006961979224608946
IMPROVEMENT: 5.685055069366612e-05
MSE for XGBoost: 0.00032146390757976055
 PERCENT IMPROVEMENT: 7.549387950488246%
---------------------------------


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


4. ADDING GDP__c3__lag_1:
Aggregated MSE: 0.0007364831191450226
IMPROVEMENT: 1.6565354009538116e-05
MSE for XGBoost: 0.0003395421350266472
 PERCENT IMPROVEMENT: 2.1997726042979613%
---------------------------------


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


5. ADDING GDP__sum_values:
Aggregated MSE: 0.0007006602163871112
IMPROVEMENT: 5.2388256767449534e-05
MSE for XGBoost: 0.0003282536760608834
 PERCENT IMPROVEMENT: 6.9568239807979815%
---------------------------------


In [15]:
import ast
import re

target = 'PRFI'
print(f"Target: {target}")
# Extract base features for the target
base_features_str = feature_engineering_results.loc[feature_engineering_results['target'] == target]['final_feature_space'].values[0]

# Convert the string representation of the list to an actual list
base_features = ast.literal_eval(base_features_str)

# Define a function to replace ""text"" with "text"
def clean_double_quotes(feature_name):
    return re.sub(r'""(.*?)""', r'"\1"', feature_name)

# Apply the cleaning function to all feature names
base_features = [clean_double_quotes(feature) for feature in base_features]

# Verify that these features exist in the tsfresh data
missing_features = [feature for feature in base_features if feature not in tsfresh_features_train.columns]
if missing_features:
    print(f"Warning: The following features are missing from the tsfresh data: {missing_features}")

# Proceed with the features that are valid
valid_features = [feature for feature in base_features if feature in tsfresh_features_train.columns]

# Now use the cleaned and verified base_features list
X_train = tsfresh_features_train[valid_features]  # Select columns matching these features
X_test = tsfresh_features_test[valid_features]  # Select columns matching these features
y_train = train_combined[[target]]
y_test = test_combined[[target]]

# Continue with MSE calculation and the rest of your logic
baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, valid_features)

#### PRINT
print("1. BASE FEATURE EVAL:")
print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")

# Feature addition steps
features_to_add = [
    'EXPGS__ratio_beyond_r_sigma__r_5', 
    'GDP__large_standard_deviation__r_0.2', 
    'PCE__minimum', 
    'GDP__symmetry_looking__r_0.05'
]

for i, feature in enumerate(features_to_add):
    base_features.append(feature)
    X_train = tsfresh_features_train[base_features]
    X_test = tsfresh_features_test[base_features]
    new_mse_scores, new_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, base_features)
    print(f"{i+2}. ADDING {feature}:")
    print(f"Aggregated MSE: {new_aggregated_mse}")
    print(f"IMPROVEMENT: {aggregated_baseline_mse - new_aggregated_mse}")
    print(f"MSE for XGBoost: {new_mse_scores['XGBoost']}")
    print(F" PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")
    print("---------------------------------")


Target: PRFI


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.0006071510422180565
Initial MSE for XGBoost: 0.0003052800622916007
Initial MSE for LightGBM: 0.0003018709799264559


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


2. ADDING EXPGS__ratio_beyond_r_sigma__r_5:
Aggregated MSE: 0.0006004040950592824
IMPROVEMENT: 6.74694715877412e-06
MSE for XGBoost: 0.0002985331151328266
 PERCENT IMPROVEMENT: 1.1112469039213102%
---------------------------------


...

3. ADDING GDP__large_standard_deviation__r_0.2:
Aggregated MSE: 0.0009569446343329441
IMPROVEMENT: -0.00034979359211488755
MSE for XGBoost: 0.00043618618305020377
 PERCENT IMPROVEMENT: -57.61228554216336%
---------------------------------


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


4. ADDING PCE__minimum:
Aggregated MSE: 0.003686326842918132
IMPROVEMENT: -0.0030791758007000757
MSE for XGBoost: 0.0008003873920329752
 PERCENT IMPROVEMENT: -507.15153011204063%
---------------------------------


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


5. ADDING GDP__symmetry_looking__r_0.05:
Aggregated MSE: 0.003617454229103949
IMPROVEMENT: -0.0030103031868858924
MSE for XGBoost: 0.0007315147782187918
 PERCENT IMPROVEMENT: -495.8079584099192%
---------------------------------


# SECTION 2:

In [ ]:
import ast
import re
from sklearn.feature_selection import mutual_info_regression
import numpy as np

def evaluate_target(target, features_to_add, handle_missing_feature='add_similar'):
    """
    Evaluate a target by adding features and calculating MSE and MSE/Variance, 
    selecting the best similar feature based on Mutual Information.

    Parameters:
    - target: The target variable.
    - features_to_add: List of features to add for evaluation.
    - handle_missing_feature: 'skip' to skip missing features, 'add_similar' to add one most informative similar feature.
    """
    print(f"Target: {target}")
    
    # Extract and clean base features
    base_features_str = feature_engineering_results.loc[feature_engineering_results['target'] == target]['final_feature_space'].values[0]
    base_features = [re.sub(r'""(.*?)""', r'"\1"', feature) for feature in ast.literal_eval(base_features_str)]
    valid_base_features = [feature for feature in base_features if feature in tsfresh_features_train.columns]
    
    # Initial MSE and Variance calculation with base features
    X_train_base = tsfresh_features_train[valid_base_features]
    X_test_base = tsfresh_features_test[valid_base_features]
    y_train = train_combined[[target]].values.ravel()  # Ensure y is 1D
    y_test = test_combined[[target]].values.ravel()    # Ensure y is 1D

    # Baseline calculations
    baseline_mse_scores, aggregated_baseline_mse, _ = compute_mse_scores(X_train_base, X_test_base, y_train, y_test, valid_base_features)
    y_train_variance = np.var(y_train)
    y_test_variance = np.var(y_test)

    baseline_mse_variance_train = baseline_mse_scores['XGBoost'] / y_train_variance if y_train_variance != 0 else np.inf
    baseline_mse_variance_test = baseline_mse_scores['XGBoost'] / y_test_variance if y_test_variance != 0 else np.inf
    
    print("1. BASE FEATURE EVAL:")
    print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
    print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
    print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")
    print(f"Initial MSE/Variance for XGBoost (Train): {baseline_mse_variance_train}")
    print(f"Initial MSE/Variance for XGBoost (Test): {baseline_mse_variance_test}")
    
    # Store baseline values for comparison
    previous_mse = aggregated_baseline_mse
    previous_mse_variance_train = baseline_mse_variance_train
    previous_mse_variance_test = baseline_mse_variance_test
    
    # Feature addition steps - isolate the effect of each feature
    for i, feature in enumerate(features_to_add):
        if feature not in tsfresh_features_train.columns:
            if handle_missing_feature == 'skip':
                print(f"Skipping missing feature: {feature}")
                continue
            elif handle_missing_feature == 'add_similar':
                # Find all features that start with the incomplete feature name
                similar_features = [col for col in tsfresh_features_train.columns if col.startswith(feature)]
                
                if similar_features:
                    # Select the best feature based on Mutual Information
                    X_train_similar = tsfresh_features_train[similar_features]
                    mi = mutual_info_regression(X_train_similar, y_train)
                    best_feature = similar_features[np.argmax(mi)]
                    print(f"Adding the most informative feature: {best_feature}")
                    valid_feature_set = valid_base_features + [best_feature]
                    
                    # Evaluate only the base features + the new feature
                    X_train = tsfresh_features_train[valid_feature_set]
                    X_test = tsfresh_features_test[valid_feature_set]
                    new_mse_scores, new_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, valid_feature_set)
                    
                    # MSE/Variance calculations
                    mse_variance_train = new_mse_scores['XGBoost'] / y_train_variance if y_train_variance != 0 else np.inf
                    mse_variance_test = new_mse_scores['XGBoost'] / y_test_variance if y_test_variance != 0 else np.inf
                    
                    # Calculate improvements
                    mse_improvement = previous_mse - new_aggregated_mse
                    mse_variance_improvement_train = previous_mse_variance_train - mse_variance_train
                    mse_variance_improvement_test = previous_mse_variance_test - mse_variance_test
                    
                    # Output the results
                    print(f"{i+2}. ADDING {best_feature} (Isolated Effect):")
                    print(f"Aggregated MSE: {new_aggregated_mse}")
                    print(f"IMPROVEMENT in MSE: {mse_improvement}")
                    print(f"MSE for XGBoost: {new_mse_scores['XGBoost']}")
                    print(f"IMPROVEMENT in MSE/Variance (Train): {mse_variance_improvement_train}")
                    print(f"IMPROVEMENT in MSE/Variance (Test): {mse_variance_improvement_test}")
                    print(f"PERCENT IMPROVEMENT: {((previous_mse - new_aggregated_mse) / previous_mse) * 100}%")
                    print("---------------------------------")
                    
                    # Update baseline values after each feature evaluation
                    previous_mse = new_aggregated_mse
                    previous_mse_variance_train = mse_variance_train
                    previous_mse_variance_test = mse_variance_test
                    
                else:
                    print(f"No similar features found for {feature}, skipping.")
                continue
        else:
            valid_feature_set = valid_base_features + [feature]
        
        # Calculate MSE after adding the feature (isolated effect)
        X_train = tsfresh_features_train[valid_feature_set]
        X_test = tsfresh_features_test[valid_feature_set]
        new_mse_scores, new_aggregated_mse, _ = compute_mse_scores(X_train, X_test, y_train, y_test, valid_feature_set)
        
        # MSE/Variance calculations
        mse_variance_train = new_mse_scores['XGBoost'] / y_train_variance if y_train_variance != 0 else np.inf
        mse_variance_test = new_mse_scores['XGBoost'] / y_test_variance if y_test_variance != 0 else np.inf
        
        # Calculate improvements
        mse_improvement = previous_mse - new_aggregated_mse
        mse_variance_improvement_train = previous_mse_variance_train - mse_variance_train
        mse_variance_improvement_test = previous_mse_variance_test - mse_variance_test
        
        # Output the results
        print(f"{i+2}. ADDING {feature} (Isolated Effect):")
        print(f"Aggregated MSE: {new_aggregated_mse}")
        print(f"IMPROVEMENT in MSE: {mse_improvement}")
        print(f"MSE for XGBoost: {new_mse_scores['XGBoost']}")
        print(f"IMPROVEMENT in MSE/Variance (Train): {mse_variance_improvement_train}")
        print(f"IMPROVEMENT in MSE/Variance (Test): {mse_variance_improvement_test}")
        print(f"MSE/Variance % Improvement (Train): {((previous_mse_variance_train - mse_variance_train) / previous_mse_variance_train) * 100}%")
        print(f"MSE/Variance % Improvement (Test): {((previous_mse_variance_test - mse_variance_test) / previous_mse_variance_test) * 100}%")
        print(f"PERCENT IMPROVEMENT: {((previous_mse - new_aggregated_mse) / previous_mse) * 100}%")
        print("---------------------------------")
        
        # Update baseline values after each feature evaluation
        previous_mse = new_aggregated_mse
        previous_mse_variance_train = mse_variance_train
        previous_mse_variance_test = mse_variance_test


In [46]:
# Combined Target Evaluations (Corrected)

# EXPGS
evaluate_target('EXPGS', [
    'PNFI__longest_strike_below_mean', 
    'DGS2__number_peaks__n_10',
    'DGS10__energy_ratio_by_chunks__num_segments_10__segment_focus_8',
    'PAYEMS__cid_ce__normalize_True',
    'PCEPILFE__number_peaks__n_50',
    'CUMFNS__change_quantiles__f_agg_',
    'EXPGS__change_quantiles__f_agg_',
    'IMPGS__energy_ratio_by_chunks__num_segments_10__segment_focus_0',
    'INDPRO__change_quantiles__f_agg_',
    'PAYEMS__cwt_coefficients__coeff_0__w_20__widths_',
    'PAYEMS__index_mass_quantile__q_0.2',
    'PRFI__fft_coefficient__attr_'
])

# HOUST
evaluate_target('HOUST', [
    'PCEPI__quantile__q_0.4',
    'GDP__percentage_of_reoccurring_values_to_all_values',
    'GDP__percentage_of_reoccurring_datapoints_to_all_datapoints',
    'GDP__ratio_value_number_to_time_series_length',
    'HOUST__last_location_of_maximum',
    'HOUST__last_location_of_minimum',
    'HOUST__change_quantiles__f_agg_'
])

# DSPI
evaluate_target('DSPI', [
    'UNRATE__approximate_entropy__m_2__r_0.1',
    'EXPGS__lempel_ziv_complexity__bins_3',
    'PAYEMS__energy_ratio_by_chunks__num_segments_10__segment_focus_0',
    'IMPGS__lempel_ziv_complexity__bins_3',
    'DGS10__percentage_of_reoccurring_values_to_all_values',
    'AAA__agg_autocorrelation__f_agg_',
    'CPILFESL__change_quantiles__f_agg_',
    'CUSR0000SAH1__approximate_entropy__m_2__r_0.3',
    'DGS5__approximate_entropy__m_2__r_0.5',
    'DSPI__change_quantiles__f_agg_',
    'DSPI__last_location_of_maximum',
    'GDP__change_quantiles__f_agg_',
    'IMPGS__lempel_ziv_complexity__bins_10',
    'INDPRO__last_location_of_maximum',
    'PCE__energy_ratio_by_chunks__num_segments_10__segment_focus_0'
])

# DGS5
evaluate_target('DGS5', [
    'PNFI__large_standard_deviation__r_0.15000000000000002',
    'BAA__change_quantiles__f_agg_',
    'DGS10__fft_coefficient__attr_',
    'DGS2__fft_coefficient__attr_'
])

# DGS10
evaluate_target('DGS10', [
    'CPILFESL__longest_strike_below_mean',
    'AAA__change_quantiles__f_agg_',
    'AAA__cid_ce__normalize_False',
    'CPIAUCSL__change_quantiles__f_agg_',
    'DGS10__change_quantiles__f_agg_',
    'DGS2__change_quantiles__f_agg_',
    'DGS5__change_quantiles__f_agg_',
    'DSPI__fft_coefficient__attr_',
    'EXPGS__change_quantiles__f_agg_',
    'GDP__change_quantiles__f_agg_',
    'PRFI__change_quantiles__f_agg_'
])

# AAA
evaluate_target('AAA', [
    'DGS10__time_reversal_asymmetry_statistic__lag_2',
    'GDP__quantile__q_0.2',
    'GDP__quantile__q_0.4',
    'GDPCTPI__count_below__t_0',
    'EXPGS__sum_of_reoccurring_values',
    'FGCE__c3__lag_1',
    'AAA__fft_coefficient__attr_',
    'BAA__ratio_beyond_r_sigma__r_2',
    'PAYEMS__mean'
])

# BAA
evaluate_target('BAA', [
    'CUSR0000SAH1__symmetry_looking__r_0.15000000000000002',
    'DGS2__symmetry_looking__r_0.2',
    'CPILFESL__maximum',
    'PAYEMS__large_standard_deviation__r_0.15000000000000002',
    'UNRATE__large_standard_deviation__r_0.2',
    'GDP__number_peaks__n_10',
    'GDPCTPI__number_peaks__n_50',
    'DGS10__cwt_coefficients__coeff_0__w_20__widths_',
    'DGS5__lempel_ziv_complexity__bins_100',
    'PRFI__change_quantiles__f_agg_',
    'WTISPLC__change_quantiles__f_agg_',
    'WTISPLC__time_reversal_asymmetry_statistic__lag_3'
])

# WTISPLC
evaluate_target('WTISPLC', [
    'DGS10__mean',
    'GCE__index_mass_quantile__q_0.4',
    'CPIAUCSL__ratio_beyond_r_sigma__r_2.5',
    'AAA__energy_ratio_by_chunks__num_segments_10__segment_focus_1',
    'BAA__energy_ratio_by_chunks__num_segments_10__segment_focus_1',
    'BAA__energy_ratio_by_chunks__num_segments_10__segment_focus_4',
    'CPIAUCSL__change_quantiles__f_agg_',
    'CPILFESL__change_quantiles__f_agg_',
    'FEDFUNDS__change_quantiles__f_agg_',
    'INDPRO__change_quantiles__f_agg_',
    'PCEPILFE__change_quantiles__f_agg_',
    'PNFI__agg_autocorrelation__f_agg_',
    'WTISPLC__change_quantiles__f_agg_',
    'WTISPLC__last_location_of_minimum'
])

# IMPGS
evaluate_target('IMPGS', [
    'PCE__last_location_of_maximum',
    'AAA__last_location_of_minimum',
    'FEDFUNDS__percentage_of_reoccurring_datapoints_to_all_datapoints',
    'PNFI__ratio_beyond_r_sigma__r_2.5',
    'CPIAUCSL__change_quantiles__f_agg_',
    'CPILFESL__ratio_beyond_r_sigma__r_5',
    'CUMFNS__change_quantiles__f_agg_',
    'HOUST__change_quantiles__f_agg_',
    'IMPGS__change_quantiles__f_agg_',
    'PAYEMS__longest_strike_above_mean',
    'PNFI__change_quantiles__f_agg_'
])

# GCE
evaluate_target('GCE', [
    'BAA__lempel_ziv_complexity__bins_2',
    'EXPGS__median',
    'UNRATE__agg_autocorrelation__f_agg_'
])

# FGCE
evaluate_target('FGCE', [
    'CUSR0000SAH1__value_count__value_0',
    'GDPCTPI__count_above__t_0',
    'AAA__change_quantiles__f_agg_',
    'CPILFESL__change_quantiles__f_agg_',
    'CPILFESL__lempel_ziv_complexity__bins_2',
    'FEDFUNDS__change_quantiles__f_agg_',
    'GDPCTPI__cwt_coefficients__coeff_0__w_5__widths_',
    'HOUST__cwt_coefficients__coeff_0__w_5__widths_',
    'PAYEMS__cwt_coefficients__coeff_0__w_5__widths_',
    'PCEPILFE__index_mass_quantile__q_0.3',
    'PNFI__cwt_coefficients__coeff_0__w_5__widths_'
])

# PCEPI
evaluate_target('PCEPI', [
    'PCEPI__sum_values',
    'GCE__last_location_of_maximum',
    'DGS5__quantile__q_0.7',
    'CPIAUCSL__last_location_of_minimum',
    'GDPCTPI__lempel_ziv_complexity__bins_10',
    'AAA__change_quantiles__f_agg_',
    'BAA__variation_coefficient',
    'FGCE__percentage_of_reoccurring_datapoints_to_all_datapoints',
    'INDPRO__lempel_ziv_complexity__bins_5',
    'PAYEMS__index_mass_quantile__q_0.9',
    'PCEPI__fft_coefficient__attr_'
])

# PCEPILFE
evaluate_target('PCEPILFE', [
    'HOUST__number_peaks__n_50',
    'DGS2__number_peaks__n_50',
    'HOUST__sum_of_reoccurring_values',
    'WTISPLC__minimum',
    'UNRATE__absolute_maximum',
    'DGS2__maximum',
    'DGS5__symmetry_looking__r_0.05',
    'DGS10__cwt_coefficients__coeff_0__w_5__widths_',
    'EXPGS__cwt_coefficients__coeff_0__w_5__widths_',
    'GDPCTPI__cwt_coefficients__coeff_0__w_5__widths_',
    'HOUST__cwt_coefficients__coeff_0__w_5__widths_',
    'INDPRO__cwt_coefficients__coeff_0__w_5__widths_',
    'PAYEMS__cwt_coefficients__coeff_0__w_5__widths_',
    'PNFI__cwt_coefficients__coeff_0__w_5__widths_',
    'PRFI__has_duplicate'
])

# PAYEMS
evaluate_target('PAYEMS', [
    'PAYEMS__index_mass_quantile__q_0.8',
    'EXPGS__lempel_ziv_complexity__bins_3',
    'DGS2__variation_coefficient',
    'GCE__mean',
    'INDPRO__lempel_ziv_complexity__bins_3',
    'HOUST__lempel_ziv_complexity__bins_5',
    'DSPI__energy_ratio_by_chunks__num_segments_10__segment_focus_9',
    'FEDFUNDS__change_quantiles__f_agg_',
    'GDP__change_quantiles__f_agg_',
    'PAYEMS__change_quantiles__f_agg_',
    'PAYEMS__last_location_of_minimum',
    'PCE__change_quantiles__f_agg_'
])

# UNRATE
evaluate_target('UNRATE', [
    'UNRATE__sum_values',
    'UNRATE__mean',
    'UNRATE__last_location_of_minimum',
    'PCEPI__first_location_of_minimum',
    'INDPRO__lempel_ziv_complexity__bins_10',
    'PCEPILFE__energy_ratio_by_chunks__num_segments_10__segment_focus_4',
    'GCE__ratio_beyond_r_sigma__r_2.5',
    'CPILFESL__count_above__t_0',
    'CPILFESL__last_location_of_maximum',
    'DSPI__agg_autocorrelation__f_agg_',
    'FEDFUNDS__change_quantiles__f_agg_'
])

# INDPRO
evaluate_target('INDPRO', [
    'INDPRO__last_location_of_maximum',
    'IMPGS__first_location_of_minimum',
    'INDPRO__variation_coefficient',
    'CPILFESL__variation_coefficient',
    'DSPI__approximate_entropy__m_2__r_0.1',
    'PAYEMS__lempel_ziv_complexity__bins_100',
    'IMPGS__energy_ratio_by_chunks__num_segments_10__segment_focus_9',
    'IMPGS__index_mass_quantile__q_0.7',
    'CUSR0000SAH1__ratio_value_number_to_time_series_length',
    'AAA__percentage_of_reoccurring_values_to_all_values',
    'INDPRO__abs_energy',
    'INDPRO__change_quantiles__f_agg_',
    'INDPRO__last_location_of_minimum'
])

# CUMFNS
evaluate_target('CUMFNS', [
    'CUMFNS__mean',
    'HOUST__lempel_ziv_complexity__bins_2',
    'DSPI__percentage_of_reoccurring_datapoints_to_all_datapoints',
    'PNFI__count_above__t_0',
    'PNFI__count_below__t_0',
    'AAA__number_peaks__n_1',
    'CUMFNS__change_quantiles__f_agg_',
    'CUMFNS__last_location_of_maximum'
])

# USREC
evaluate_target('USREC', [
    'FEDFUNDS__has_duplicate_max',
    'FEDFUNDS__has_duplicate_min',
    'FEDFUNDS__variance_larger_than_standard_deviation'
])

# FEDFUNDS
evaluate_target('FEDFUNDS', [
    'CUMFNS__has_duplicate',
    'FGCE__has_duplicate_min',
    'INDPRO__minimum',
    'FEDFUNDS__has_duplicate',
    'FEDFUNDS__minimum',
    'BAA__large_standard_deviation__r_0.2'
])

# GDP
evaluate_target('GDP', [
    'WTISPLC__index_mass_quantile__q_0.4',
    'PNFI__index_mass_quantile__q_0.9'
])

# CPIAUCSL
evaluate_target('CPIAUCSL', [
    'DGS10__longest_strike_below_mean',
    'PRFI__maximum',
    'CUMFNS__has_duplicate_min',
    'BAA__sum_of_reoccurring_values',
    'CPIAUCSL__large_standard_deviation__r_0.4'
])

# CUSR0000SAH1
evaluate_target('CUSR0000SAH1', [
    'CUMFNS__has_duplicate_min',
    'PCEPILFE__ratio_beyond_r_sigma__r_5',
    'FEDFUNDS__has_duplicate_min',
    'PAYEMS__symmetry_looking__r_0.1',
    'WTISPLC__has_duplicate'
])

# CPILFESL
evaluate_target('CPILFESL', [
    'DGS10__minimum',
    'EXPGS__symmetry_looking__r_0.1',
    'GDP__symmetry_looking__r_0.05',
    'DGS2__large_standard_deviation__r_0.30000000000000004'
])

# PCE
evaluate_target('PCE', [
    'PCE__last_location_of_minimum',
    'INDPRO__cid_ce__normalize_False',
    'GDP__c3__lag_1',
    'GDP__sum_values'
])

# PRFI
evaluate_target('PRFI', [
    'EXPGS__ratio_beyond_r_sigma__r_5',
    'GDP__large_standard_deviation__r_0.2',
    'PCE__minimum',
    'GDP__symmetry_looking__r_0.05'
])

import warnings
from sklearn.exceptions import DataConversionWarning

# Suppress DataConversionWarning specifically
warnings.filterwarnings(action='ignore', category=DataConversionWarning)


Target: EXPGS
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.0007083075575865404
Initial MSE for XGBoost: 0.0003575527559418474
Initial MSE for LightGBM: 0.00035075480164469306
2. ADDING PNFI__longest_strike_below_mean (Isolated Effect):
Aggregated MSE: 0.0007900623185914516
IMPROVEMENT: -8.175476100491118e-05
MSE for XGBoost: 0.00039210810690614204
PERCENT IMPROVEMENT: -11.542268627413657%
---------------------------------
3. ADDING DGS2__number_peaks__n_10 (Isolated Effect):
Aggregated MSE: 0.000747235832247894
IMPROVEMENT: -3.892827466135359e-05
MSE for XGBoost: 0.0003730317671891005
PERCENT IMPROVEMENT: -5.495956416728952%
---------------------------------
4. ADDING DGS10__energy_ratio_by_chunks__num_segments_10__segment_focus_8 (Isolated Effect):
Aggregated MSE: 0.0006934720418009271
IMPROVEMENT: 1.4835515785613373e-05
MSE for XGBoost: 0.00029166384422549545
PERCENT IMPROVEMENT: 2.094501975407877%
---------------------------------
5. ADDING PAYEMS__cid_ce__normalize_True

C:\Users\nabounaser\AppData\Local\Temp\ipykernel_16456\2078389699.py:78: RuntimeWarning: invalid value encountered in scalar divide
  print(f"PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")


3. ADDING EXPGS__median (Isolated Effect):
Aggregated MSE: 0.00012626151230943255
IMPROVEMENT: inf
MSE for XGBoost: 6.220417312108812e-05
PERCENT IMPROVEMENT: nan%
---------------------------------
Adding the most informative feature: UNRATE__agg_autocorrelation__f_agg_"var"__maxlag_40


C:\Users\nabounaser\AppData\Local\Temp\ipykernel_16456\2078389699.py:78: RuntimeWarning: invalid value encountered in scalar divide
  print(f"PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")


4. ADDING UNRATE__agg_autocorrelation__f_agg_"var"__maxlag_40 (Isolated Effect):
Aggregated MSE: 0.0001246875185775379
IMPROVEMENT: inf
MSE for XGBoost: 6.265131535011391e-05
PERCENT IMPROVEMENT: nan%
---------------------------------
Target: FGCE


C:\Users\nabounaser\AppData\Local\Temp\ipykernel_16456\2078389699.py:61: RuntimeWarning: invalid value encountered in scalar divide
  print(f"PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")


1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.0005295874028754448
Initial MSE for XGBoost: 0.00027705804465465655
Initial MSE for LightGBM: 0.0002525293582207882
2. ADDING CUSR0000SAH1__value_count__value_0 (Isolated Effect):
Aggregated MSE: 0.0005873599535917745
IMPROVEMENT: -5.777255071632973e-05
MSE for XGBoost: 0.00034813351081701746
PERCENT IMPROVEMENT: -10.908973741189502%
---------------------------------
3. ADDING GDPCTPI__count_above__t_0 (Isolated Effect):
Aggregated MSE: 0.000635600167803746
IMPROVEMENT: -0.0001060127649283012
MSE for XGBoost: 0.0003436644787300224
PERCENT IMPROVEMENT: -20.017992186501207%
---------------------------------
Adding the most informative feature: AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0
4. ADDING AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0 (Isolated Effect):
Aggregated MSE: 0.0011079978374391054
IMPROVEMENT: -0.0005784104345636606
MSE for XGBoost: 0.00028233463217142716
PERCENT IMPROVEMENT: -10

In [36]:
text  = r"""
Target: EXPGS
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.0007083075575865404
Initial MSE for XGBoost: 0.0003575527559418474
Initial MSE for LightGBM: 0.00035075480164469306
2. ADDING PNFI__longest_strike_below_mean:
Aggregated MSE: 0.0007900623185914516
IMPROVEMENT: -8.175476100491118e-05
MSE for XGBoost: 0.00039210810690614204
PERCENT IMPROVEMENT: -11.542268627413657%
---------------------------------
3. ADDING DGS2__number_peaks__n_10:
Aggregated MSE: 0.0007621176066378139
IMPROVEMENT: -5.381004905127341e-05
MSE for XGBoost: 0.0003613206884182433
PERCENT IMPROVEMENT: -7.596989256280658%
---------------------------------
4. ADDING DGS10__energy_ratio_by_chunks__num_segments_10__segment_focus_8:
Aggregated MSE: 0.0007937651825563744
IMPROVEMENT: -8.545762496983396e-05
MSE for XGBoost: 0.00034976976947832344
PERCENT IMPROVEMENT: -12.065044916507588%
---------------------------------
5. ADDING PAYEMS__cid_ce__normalize_True:
Aggregated MSE: 0.0007386971039901024
IMPROVEMENT: -3.038954640356196e-05
MSE for XGBoost: 0.00029631364678177867
PERCENT IMPROVEMENT: -4.290445030278954%
---------------------------------
6. ADDING PCEPILFE__number_peaks__n_50:
Aggregated MSE: 0.0007828342301071988
IMPROVEMENT: -7.45266725206583e-05
MSE for XGBoost: 0.0002994265310950703
PERCENT IMPROVEMENT: -10.52179547181419%
---------------------------------
Adding all similar features for CUMFNS__change_quantiles__f_agg_: ['CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for EXPGS__change_quantiles__f_agg_: ['EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
9. ADDING IMPGS__energy_ratio_by_chunks__num_segments_10__segment_focus_0:
Aggregated MSE: 0.002410743758402365
IMPROVEMENT: -0.0017024362008158248
MSE for XGBoost: 0.0010617209072010735
PERCENT IMPROVEMENT: -240.3526804961166%
---------------------------------
Adding all similar features for INDPRO__change_quantiles__f_agg_: ['INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for PAYEMS__cwt_coefficients__coeff_0__w_20__widths_: ['PAYEMS__cwt_coefficients__coeff_0__w_20__widths_(2, 5, 10, 20)']
12. ADDING PAYEMS__index_mass_quantile__q_0.2:
Aggregated MSE: 0.0030058422349792235
IMPROVEMENT: -0.002297534677392683
MSE for XGBoost: 0.0014445544679534562
PERCENT IMPROVEMENT: -324.3696403891571%
---------------------------------
Adding all similar features for PRFI__fft_coefficient__attr_: ['PRFI__fft_coefficient__attr_"real"__coeff_0', 'PRFI__fft_coefficient__attr_"imag"__coeff_0', 'PRFI__fft_coefficient__attr_"abs"__coeff_0', 'PRFI__fft_coefficient__attr_"angle"__coeff_0']
Target: HOUST
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 21.5155204404318
Initial MSE for XGBoost: 10.905371214554187
Initial MSE for LightGBM: 10.610149225877613
2. ADDING PCEPI__quantile__q_0.4:
Aggregated MSE: 21.27976422095753
IMPROVEMENT: 0.2357562194742684
MSE for XGBoost: 10.669614995079918
PERCENT IMPROVEMENT: 1.0957495549642253%
---------------------------------
3. ADDING GDP__percentage_of_reoccurring_values_to_all_values:
Aggregated MSE: 20.98774265229224
IMPROVEMENT: 0.5277777881395593
MSE for XGBoost: 10.504441356373345
PERCENT IMPROVEMENT: 2.4530096290293004%
---------------------------------
4. ADDING GDP__percentage_of_reoccurring_datapoints_to_all_datapoints:
Aggregated MSE: 21.01026193326365
IMPROVEMENT: 0.5052585071681506
MSE for XGBoost: 10.50653807515218
PERCENT IMPROVEMENT: 2.3483443431778332%
---------------------------------
5. ADDING GDP__ratio_value_number_to_time_series_length:
Aggregated MSE: 21.02354589954033
IMPROVEMENT: 0.4919745408914693
MSE for XGBoost: 10.473420507041062
PERCENT IMPROVEMENT: 2.2866030234014447%
---------------------------------
6. ADDING HOUST__last_location_of_maximum:
Aggregated MSE: 21.055777664324616
IMPROVEMENT: 0.45974277610718417
MSE for XGBoost: 10.504880038073432
PERCENT IMPROVEMENT: 2.13679598120824%
---------------------------------
7. ADDING HOUST__last_location_of_minimum:
Aggregated MSE: 21.316017112964246
IMPROVEMENT: 0.1995033274675535
MSE for XGBoost: 10.598588437887239
PERCENT IMPROVEMENT: 0.9272530869978325%
---------------------------------
Adding all similar features for HOUST__change_quantiles__f_agg_: ['HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Target: DSPI
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.0018684591867704492
Initial MSE for XGBoost: 0.0009345199138528737
Initial MSE for LightGBM: 0.0009339392729175755
2. ADDING UNRATE__approximate_entropy__m_2__r_0.1:
Aggregated MSE: 0.001871830100426572
IMPROVEMENT: -3.3709136561226993e-06
MSE for XGBoost: 0.0009385916751874036
PERCENT IMPROVEMENT: -0.1804114149236076%
---------------------------------
3. ADDING EXPGS__lempel_ziv_complexity__bins_3:
Aggregated MSE: 0.0019125781806925255
IMPROVEMENT: -4.411899392207626e-05
MSE for XGBoost: 0.0009523480809993186
PERCENT IMPROVEMENT: -2.3612500735610946%
---------------------------------
4. ADDING PAYEMS__energy_ratio_by_chunks__num_segments_10__segment_focus_0:
Aggregated MSE: 0.0018978313863508506
IMPROVEMENT: -2.9372199580401416e-05
MSE for XGBoost: 0.0009397283250669545
PERCENT IMPROVEMENT: -1.5720011327178085%
---------------------------------
5. ADDING IMPGS__lempel_ziv_complexity__bins_3:
Aggregated MSE: 0.0019411292704050078
IMPROVEMENT: -7.267008363455864e-05
MSE for XGBoost: 0.000986525165385811
PERCENT IMPROVEMENT: -3.8893053778801416%
---------------------------------
6. ADDING DGS10__percentage_of_reoccurring_values_to_all_values:
Aggregated MSE: 0.0019552061540391193
IMPROVEMENT: -8.674696726867007e-05
MSE for XGBoost: 0.0009646272154096114
PERCENT IMPROVEMENT: -4.6427006745921195%
---------------------------------
Adding all similar features for AAA__agg_autocorrelation__f_agg_: ['AAA__agg_autocorrelation__f_agg_"mean"__maxlag_40', 'AAA__agg_autocorrelation__f_agg_"median"__maxlag_40', 'AAA__agg_autocorrelation__f_agg_"var"__maxlag_40']
Adding all similar features for CPILFESL__change_quantiles__f_agg_: ['CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
9. ADDING CUSR0000SAH1__approximate_entropy__m_2__r_0.3:
Aggregated MSE: 0.002939455204489334
IMPROVEMENT: -0.0010709960177188848
MSE for XGBoost: 0.0012696306511769074
PERCENT IMPROVEMENT: -57.319743738692786%
---------------------------------
10. ADDING DGS5__approximate_entropy__m_2__r_0.5:
Aggregated MSE: 0.0027984796424934434
IMPROVEMENT: -0.0009300204557229942
MSE for XGBoost: 0.0012738964738861326
PERCENT IMPROVEMENT: -49.7747268074126%
---------------------------------
Adding all similar features for DSPI__change_quantiles__f_agg_: ['DSPI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'DSPI__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'DSPI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'DSPI__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'DSPI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'DSPI__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'DSPI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'DSPI__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'DSPI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'DSPI__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'DSPI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'DSPI__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'DSPI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'DSPI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'DSPI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'DSPI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'DSPI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'DSPI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'DSPI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'DSPI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'DSPI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'DSPI__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'DSPI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'DSPI__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'DSPI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'DSPI__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'DSPI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'DSPI__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'DSPI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'DSPI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'DSPI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'DSPI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'DSPI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'DSPI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'DSPI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'DSPI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'DSPI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'DSPI__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'DSPI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'DSPI__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'DSPI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'DSPI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'DSPI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'DSPI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'DSPI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'DSPI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'DSPI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'DSPI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'DSPI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'DSPI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'DSPI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'DSPI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'DSPI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'DSPI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'DSPI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'DSPI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'DSPI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'DSPI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'DSPI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'DSPI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
12. ADDING DSPI__last_location_of_maximum:
Aggregated MSE: 0.0032368081588376486
IMPROVEMENT: -0.0013683489720671994
MSE for XGBoost: 0.0015041659258499206
PERCENT IMPROVEMENT: -73.23408409216213%
---------------------------------
Adding all similar features for GDP__change_quantiles__f_agg_: ['GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
14. ADDING IMPGS__lempel_ziv_complexity__bins_10:
Aggregated MSE: 0.003023426829007526
IMPROVEMENT: -0.0011549676422370767
MSE for XGBoost: 0.0014426120731521066
PERCENT IMPROVEMENT: -61.81390797373467%
---------------------------------
15. ADDING INDPRO__last_location_of_maximum:
Aggregated MSE: 0.003036913264688605
IMPROVEMENT: -0.0011684540779181556
MSE for XGBoost: 0.0014397939517021077
PERCENT IMPROVEMENT: -62.53570247567344%
---------------------------------
16. ADDING PCE__energy_ratio_by_chunks__num_segments_10__segment_focus_0:
Aggregated MSE: 0.003048753061880025
IMPROVEMENT: -0.0011802938751095757
MSE for XGBoost: 0.0014546669324426065
PERCENT IMPROVEMENT: -63.169368828958085%
---------------------------------
Target: DGS5
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.014415792166031855
Initial MSE for XGBoost: 0.0066586914827424844
Initial MSE for LightGBM: 0.00775710068328937
2. ADDING PNFI__large_standard_deviation__r_0.15000000000000002:
Aggregated MSE: 0.014302160882829075
IMPROVEMENT: 0.00011363128320278013
MSE for XGBoost: 0.0066231972011447
PERCENT IMPROVEMENT: 0.788241685882037%
---------------------------------
Adding all similar features for BAA__change_quantiles__f_agg_: ['BAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'BAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'BAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'BAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'BAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'BAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'BAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'BAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'BAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'BAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'BAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'BAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'BAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'BAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'BAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'BAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'BAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'BAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'BAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'BAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'BAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'BAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'BAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'BAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'BAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'BAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'BAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'BAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'BAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'BAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'BAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'BAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'BAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'BAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'BAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'BAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'BAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'BAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'BAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'BAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'BAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'BAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'BAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'BAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'BAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'BAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'BAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'BAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'BAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'BAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'BAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'BAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'BAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'BAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'BAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'BAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'BAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'BAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'BAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'BAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for DGS10__fft_coefficient__attr_: ['DGS10__fft_coefficient__attr_"real"__coeff_0', 'DGS10__fft_coefficient__attr_"imag"__coeff_0', 'DGS10__fft_coefficient__attr_"abs"__coeff_0', 'DGS10__fft_coefficient__attr_"angle"__coeff_0']
Adding all similar features for DGS2__fft_coefficient__attr_: ['DGS2__fft_coefficient__attr_"real"__coeff_0', 'DGS2__fft_coefficient__attr_"imag"__coeff_0', 'DGS2__fft_coefficient__attr_"abs"__coeff_0', 'DGS2__fft_coefficient__attr_"angle"__coeff_0']
Target: DGS10
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.014098785216531047
Initial MSE for XGBoost: 0.0038384658625532486
Initial MSE for LightGBM: 0.010260319353977799
2. ADDING CPILFESL__longest_strike_below_mean:
Aggregated MSE: 0.011193372934674449
IMPROVEMENT: 0.0029054122818565984
MSE for XGBoost: 0.004094068790194421
PERCENT IMPROVEMENT: 20.607536303552994%
---------------------------------
Adding all similar features for AAA__change_quantiles__f_agg_: ['AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
4. ADDING AAA__cid_ce__normalize_False:
Aggregated MSE: 0.028624723606826392
IMPROVEMENT: -0.014525938390295345
MSE for XGBoost: 0.010540325531217842
PERCENT IMPROVEMENT: -103.02971615783929%
---------------------------------
Adding all similar features for CPIAUCSL__change_quantiles__f_agg_: ['CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for DGS10__change_quantiles__f_agg_: ['DGS10__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'DGS10__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'DGS10__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'DGS10__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'DGS10__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'DGS10__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'DGS10__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'DGS10__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'DGS10__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'DGS10__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'DGS10__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'DGS10__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'DGS10__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'DGS10__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'DGS10__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'DGS10__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'DGS10__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'DGS10__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'DGS10__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'DGS10__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'DGS10__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'DGS10__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'DGS10__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'DGS10__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'DGS10__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'DGS10__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'DGS10__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'DGS10__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'DGS10__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'DGS10__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'DGS10__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'DGS10__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'DGS10__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'DGS10__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'DGS10__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'DGS10__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'DGS10__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'DGS10__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'DGS10__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'DGS10__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'DGS10__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'DGS10__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'DGS10__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'DGS10__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'DGS10__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'DGS10__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'DGS10__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'DGS10__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'DGS10__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'DGS10__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'DGS10__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'DGS10__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'DGS10__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'DGS10__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'DGS10__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'DGS10__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'DGS10__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'DGS10__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'DGS10__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'DGS10__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for DGS2__change_quantiles__f_agg_: ['DGS2__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'DGS2__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'DGS2__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'DGS2__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'DGS2__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'DGS2__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'DGS2__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'DGS2__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'DGS2__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'DGS2__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'DGS2__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'DGS2__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'DGS2__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'DGS2__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'DGS2__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'DGS2__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'DGS2__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'DGS2__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'DGS2__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'DGS2__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'DGS2__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'DGS2__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'DGS2__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'DGS2__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'DGS2__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'DGS2__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'DGS2__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'DGS2__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'DGS2__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'DGS2__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'DGS2__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'DGS2__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'DGS2__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'DGS2__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'DGS2__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'DGS2__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'DGS2__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'DGS2__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'DGS2__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'DGS2__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'DGS2__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'DGS2__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'DGS2__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'DGS2__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'DGS2__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'DGS2__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'DGS2__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'DGS2__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'DGS2__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'DGS2__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'DGS2__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'DGS2__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'DGS2__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'DGS2__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'DGS2__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'DGS2__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'DGS2__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'DGS2__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'DGS2__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'DGS2__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for DGS5__change_quantiles__f_agg_: ['DGS5__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'DGS5__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'DGS5__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'DGS5__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'DGS5__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'DGS5__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'DGS5__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'DGS5__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'DGS5__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'DGS5__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'DGS5__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'DGS5__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'DGS5__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'DGS5__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'DGS5__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'DGS5__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'DGS5__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'DGS5__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'DGS5__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'DGS5__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'DGS5__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'DGS5__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'DGS5__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'DGS5__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'DGS5__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'DGS5__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'DGS5__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'DGS5__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'DGS5__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'DGS5__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'DGS5__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'DGS5__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'DGS5__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'DGS5__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'DGS5__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'DGS5__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'DGS5__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'DGS5__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'DGS5__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'DGS5__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'DGS5__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'DGS5__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'DGS5__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'DGS5__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'DGS5__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'DGS5__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'DGS5__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'DGS5__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'DGS5__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'DGS5__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'DGS5__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'DGS5__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'DGS5__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'DGS5__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'DGS5__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'DGS5__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'DGS5__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'DGS5__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'DGS5__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'DGS5__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for DSPI__fft_coefficient__attr_: ['DSPI__fft_coefficient__attr_"real"__coeff_0', 'DSPI__fft_coefficient__attr_"imag"__coeff_0', 'DSPI__fft_coefficient__attr_"abs"__coeff_0', 'DSPI__fft_coefficient__attr_"angle"__coeff_0']
Adding all similar features for EXPGS__change_quantiles__f_agg_: ['EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'EXPGS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'EXPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'EXPGS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for GDP__change_quantiles__f_agg_: ['GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for PRFI__change_quantiles__f_agg_: ['PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Target: AAA
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.008254629133549069
Initial MSE for XGBoost: 0.003750892710130511
Initial MSE for LightGBM: 0.0045037364234185575
2. ADDING DGS10__time_reversal_asymmetry_statistic__lag_2:
Aggregated MSE: 0.021321725807867287
IMPROVEMENT: -0.013067096674318218
MSE for XGBoost: 0.01634827919228691
PERCENT IMPROVEMENT: -158.30022721687115%
---------------------------------
3. ADDING GDP__quantile__q_0.2:
Aggregated MSE: 0.017388112728070778
IMPROVEMENT: -0.00913348359452171
MSE for XGBoost: 0.012414666112490402
PERCENT IMPROVEMENT: -110.64680734596222%
---------------------------------
4. ADDING GDP__quantile__q_0.4:
Aggregated MSE: 0.0145481906838769
IMPROVEMENT: -0.0062935615503278315
MSE for XGBoost: 0.009351736140488379
PERCENT IMPROVEMENT: -76.24281416531576%
---------------------------------
5. ADDING GDPCTPI__count_below__t_0:
Aggregated MSE: 0.012262499655544025
IMPROVEMENT: -0.004007870521994956
MSE for XGBoost: 0.005948876013861235
PERCENT IMPROVEMENT: -48.5530053155977%
---------------------------------
6. ADDING EXPGS__sum_of_reoccurring_values:
Aggregated MSE: 0.010914381560196765
IMPROVEMENT: -0.002659752426647696
MSE for XGBoost: 0.003380927957975579
PERCENT IMPROVEMENT: -32.22134372866899%
---------------------------------
7. ADDING FGCE__c3__lag_1:
Aggregated MSE: 0.008773209226513916
IMPROVEMENT: -0.000518580092964847
MSE for XGBoost: 0.003741766121437843
PERCENT IMPROVEMENT: -6.282294268766064%
---------------------------------
Adding all similar features for AAA__fft_coefficient__attr_: ['AAA__fft_coefficient__attr_"real"__coeff_0', 'AAA__fft_coefficient__attr_"imag"__coeff_0', 'AAA__fft_coefficient__attr_"abs"__coeff_0', 'AAA__fft_coefficient__attr_"angle"__coeff_0']
9. ADDING BAA__ratio_beyond_r_sigma__r_2:
Aggregated MSE: 0.01913268807795892
IMPROVEMENT: -0.01087805894440985
MSE for XGBoost: 0.004951408262858946
PERCENT IMPROVEMENT: -131.78131650031915%
---------------------------------
10. ADDING PAYEMS__mean:
Aggregated MSE: 0.018828053859004396
IMPROVEMENT: -0.010573424725455327
MSE for XGBoost: 0.005006558267610103
PERCENT IMPROVEMENT: -128.09085125922908%
---------------------------------
Target: BAA
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.006954154956485399
Initial MSE for XGBoost: 0.0031264010903663464
Initial MSE for LightGBM: 0.003827753866119053
2. ADDING CUSR0000SAH1__symmetry_looking__r_0.15000000000000002:
Aggregated MSE: 0.007155612524106559
IMPROVEMENT: -0.00020145756762115965
MSE for XGBoost: 0.0033278586579875064
PERCENT IMPROVEMENT: -2.8969381453498046%
---------------------------------
3. ADDING DGS2__symmetry_looking__r_0.2:
Aggregated MSE: 0.006925250850143731
IMPROVEMENT: 2.8904106341668713e-05
MSE for XGBoost: 0.003097496984024677
PERCENT IMPROVEMENT: 0.415637939081483%
---------------------------------
4. ADDING CPILFESL__maximum:
Aggregated MSE: 0.006912995640285296
IMPROVEMENT: 4.115931620010324e-05
MSE for XGBoost: 0.003085241774166243
PERCENT IMPROVEMENT: 0.5918665381725257%
---------------------------------
5. ADDING PAYEMS__large_standard_deviation__r_0.15000000000000002:
Aggregated MSE: 0.006975085465867238
IMPROVEMENT: -2.0930509381838526e-05
MSE for XGBoost: 0.003063162435611309
PERCENT IMPROVEMENT: -0.30097847276639805%
---------------------------------
6. ADDING UNRATE__large_standard_deviation__r_0.2:
Aggregated MSE: 0.007140961563323319
IMPROVEMENT: -0.00018680660683791962
MSE for XGBoost: 0.0032972041307089647
PERCENT IMPROVEMENT: -2.6862589057453343%
---------------------------------
7. ADDING GDP__number_peaks__n_10:
Aggregated MSE: 0.0075353350892818464
IMPROVEMENT: -0.0005811801327964472
MSE for XGBoost: 0.0033348599927186693
PERCENT IMPROVEMENT: -8.357307774030005%
---------------------------------
8. ADDING GDPCTPI__number_peaks__n_50:
Aggregated MSE: 0.007255189735375965
IMPROVEMENT: -0.00030103477889056613
MSE for XGBoost: 0.003083853682386967
PERCENT IMPROVEMENT: -4.32884772879303%
---------------------------------
Adding all similar features for DGS10__cwt_coefficients__coeff_0__w_20__widths_: ['DGS10__cwt_coefficients__coeff_0__w_20__widths_(2, 5, 10, 20)']
10. ADDING DGS5__lempel_ziv_complexity__bins_100:
Aggregated MSE: 0.007276055801085762
IMPROVEMENT: -0.0003219008446003627
MSE for XGBoost: 0.0033455650209938542
PERCENT IMPROVEMENT: -4.6288995084896705%
---------------------------------
Adding all similar features for PRFI__change_quantiles__f_agg_: ['PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'PRFI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'PRFI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'PRFI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'PRFI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for WTISPLC__change_quantiles__f_agg_: ['WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
13. ADDING WTISPLC__time_reversal_asymmetry_statistic__lag_3:
Aggregated MSE: 0.1290433277509293
IMPROVEMENT: -0.12208917279444391
MSE for XGBoost: 0.06850732835804646
PERCENT IMPROVEMENT: -1755.6291678629386%
---------------------------------
Target: WTISPLC
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 28.460910267088835
Initial MSE for XGBoost: 14.25200631310089
Initial MSE for LightGBM: 14.208903953987946
2. ADDING DGS10__mean:
Aggregated MSE: 28.46257251573908
IMPROVEMENT: -0.0016622486502448908
MSE for XGBoost: 14.244593831830237
PERCENT IMPROVEMENT: -0.005840462004362014%
---------------------------------
3. ADDING GCE__index_mass_quantile__q_0.4:
Aggregated MSE: 28.457025238917044
IMPROVEMENT: 0.0038850281717905943
MSE for XGBoost: 14.248227613660104
PERCENT IMPROVEMENT: 0.013650400269463975%
---------------------------------
4. ADDING CPIAUCSL__ratio_beyond_r_sigma__r_2.5:
Aggregated MSE: 28.451202408995172
IMPROVEMENT: 0.00970785809366248
MSE for XGBoost: 14.239163690513333
PERCENT IMPROVEMENT: 0.034109443452651246%
---------------------------------
5. ADDING AAA__energy_ratio_by_chunks__num_segments_10__segment_focus_1:
Aggregated MSE: 28.466758495773107
IMPROVEMENT: -0.005848228684271817
MSE for XGBoost: 14.25345509040473
PERCENT IMPROVEMENT: -0.020548284047803265%
---------------------------------
6. ADDING BAA__energy_ratio_by_chunks__num_segments_10__segment_focus_1:
Aggregated MSE: 28.456004262998427
IMPROVEMENT: 0.004906004090408089
MSE for XGBoost: 14.244648292001722
PERCENT IMPROVEMENT: 0.01723769213411707%
---------------------------------
7. ADDING BAA__energy_ratio_by_chunks__num_segments_10__segment_focus_4:
Aggregated MSE: 28.456284734846946
IMPROVEMENT: 0.0046255322418886635
MSE for XGBoost: 14.250469266816006
PERCENT IMPROVEMENT: 0.016252228753334926%
---------------------------------
Adding all similar features for CPIAUCSL__change_quantiles__f_agg_: ['CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for CPILFESL__change_quantiles__f_agg_: ['CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for FEDFUNDS__change_quantiles__f_agg_: ['FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for INDPRO__change_quantiles__f_agg_: ['INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for PCEPILFE__change_quantiles__f_agg_: ['PCEPILFE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'PCEPILFE__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'PCEPILFE__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for PNFI__agg_autocorrelation__f_agg_: ['PNFI__agg_autocorrelation__f_agg_"mean"__maxlag_40', 'PNFI__agg_autocorrelation__f_agg_"median"__maxlag_40', 'PNFI__agg_autocorrelation__f_agg_"var"__maxlag_40']
Adding all similar features for WTISPLC__change_quantiles__f_agg_: ['WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'WTISPLC__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'WTISPLC__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
15. ADDING WTISPLC__last_location_of_minimum:
Aggregated MSE: 28.75069063326734
IMPROVEMENT: -0.2897803661785048
MSE for XGBoost: 14.402590709987114
PERCENT IMPROVEMENT: -1.0181697052521763%
---------------------------------
Target: IMPGS
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.0001520094960760273
Initial MSE for XGBoost: 8.291012637231123e-05
Initial MSE for LightGBM: 6.909936970371606e-05
2. ADDING PCE__last_location_of_maximum:
Aggregated MSE: 0.00014195625788379033
IMPROVEMENT: 1.0053238192236978e-05
MSE for XGBoost: 7.285688818007426e-05
PERCENT IMPROVEMENT: 6.61355931816843%
---------------------------------
3. ADDING AAA__last_location_of_minimum:
Aggregated MSE: 0.0001546198251021449
IMPROVEMENT: -2.6103290261176074e-06
MSE for XGBoost: 6.703967653129304e-05
PERCENT IMPROVEMENT: -1.7172144461370067%
---------------------------------
4. ADDING FEDFUNDS__percentage_of_reoccurring_datapoints_to_all_datapoints:
Aggregated MSE: 0.00014015188775067842
IMPROVEMENT: 1.185760832534888e-05
MSE for XGBoost: 6.452880642100638e-05
PERCENT IMPROVEMENT: 7.800570774485245%
---------------------------------
5. ADDING PNFI__ratio_beyond_r_sigma__r_2.5:
Aggregated MSE: 0.00012489240391896597
IMPROVEMENT: 2.7117092157061338e-05
MSE for XGBoost: 5.9783680950559395e-05
PERCENT IMPROVEMENT: 17.839077726761733%
---------------------------------
Adding all similar features for CPIAUCSL__change_quantiles__f_agg_: ['CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'CPIAUCSL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
7. ADDING CPILFESL__ratio_beyond_r_sigma__r_5:
Aggregated MSE: 0.0008528288978026937
IMPROVEMENT: -0.0007008194017266664
MSE for XGBoost: 0.0005123552171590001
PERCENT IMPROVEMENT: -461.0365929876859%
---------------------------------
Adding all similar features for CUMFNS__change_quantiles__f_agg_: ['CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for HOUST__change_quantiles__f_agg_: ['HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'HOUST__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'HOUST__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'HOUST__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'HOUST__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for IMPGS__change_quantiles__f_agg_: ['IMPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'IMPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'IMPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'IMPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'IMPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'IMPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'IMPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'IMPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'IMPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'IMPGS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'IMPGS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'IMPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'IMPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'IMPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'IMPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'IMPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'IMPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'IMPGS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'IMPGS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'IMPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'IMPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'IMPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'IMPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'IMPGS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'IMPGS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'IMPGS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'IMPGS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'IMPGS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'IMPGS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'IMPGS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'IMPGS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'IMPGS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
11. ADDING PAYEMS__longest_strike_above_mean:
Aggregated MSE: 0.0008854926636599368
IMPROVEMENT: -0.0007334831675839095
MSE for XGBoost: 0.0004908233802710442
PERCENT IMPROVEMENT: -482.5245701867593%
---------------------------------
Adding all similar features for PNFI__change_quantiles__f_agg_: ['PNFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'PNFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'PNFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'PNFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'PNFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'PNFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'PNFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'PNFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'PNFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'PNFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'PNFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'PNFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'PNFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'PNFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'PNFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'PNFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'PNFI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'PNFI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'PNFI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'PNFI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'PNFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'PNFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'PNFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'PNFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'PNFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'PNFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'PNFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'PNFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'PNFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'PNFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'PNFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'PNFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'PNFI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'PNFI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'PNFI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'PNFI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'PNFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'PNFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'PNFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'PNFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'PNFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'PNFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'PNFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'PNFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'PNFI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'PNFI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'PNFI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'PNFI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'PNFI__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'PNFI__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'PNFI__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'PNFI__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'PNFI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'PNFI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'PNFI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'PNFI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'PNFI__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'PNFI__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'PNFI__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'PNFI__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Target: GCE
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: inf
Initial MSE for XGBoost: inf
Initial MSE for LightGBM: inf
2. ADDING BAA__lempel_ziv_complexity__bins_2:
Aggregated MSE: 0.00012410934809284195
IMPROVEMENT: inf
MSE for XGBoost: 5.924578655390218e-05
PERCENT IMPROVEMENT: nan%
---------------------------------
C:\Users\nabounaser\AppData\Local\Temp\ipykernel_16456\2914011175.py:59: RuntimeWarning: invalid value encountered in scalar divide
  print(f"PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")
3. ADDING EXPGS__median:
Aggregated MSE: 0.00011780996363640801
IMPROVEMENT: inf
MSE for XGBoost: 5.423775630060534e-05
PERCENT IMPROVEMENT: nan%
---------------------------------
Adding all similar features for UNRATE__agg_autocorrelation__f_agg_: ['UNRATE__agg_autocorrelation__f_agg_"mean"__maxlag_40', 'UNRATE__agg_autocorrelation__f_agg_"median"__maxlag_40', 'UNRATE__agg_autocorrelation__f_agg_"var"__maxlag_40']
Target: FGCE
C:\Users\nabounaser\AppData\Local\Temp\ipykernel_16456\2914011175.py:59: RuntimeWarning: invalid value encountered in scalar divide
  print(f"PERCENT IMPROVEMENT: {((aggregated_baseline_mse - new_aggregated_mse) / aggregated_baseline_mse) * 100}%")
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.0005295874028754448
Initial MSE for XGBoost: 0.00027705804465465655
Initial MSE for LightGBM: 0.0002525293582207882
2. ADDING CUSR0000SAH1__value_count__value_0:
Aggregated MSE: 0.0005873599535917745
IMPROVEMENT: -5.777255071632973e-05
MSE for XGBoost: 0.00034813351081701746
PERCENT IMPROVEMENT: -10.908973741189502%
---------------------------------
3. ADDING GDPCTPI__count_above__t_0:
Aggregated MSE: 0.0005562624689979074
IMPROVEMENT: -2.6675066122462633e-05
MSE for XGBoost: 0.0002643267799241838
PERCENT IMPROVEMENT: -5.036952536564851%
---------------------------------
Adding all similar features for AAA__change_quantiles__f_agg_: ['AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for CPILFESL__change_quantiles__f_agg_: ['CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'CPILFESL__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'CPILFESL__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
6. ADDING CPILFESL__lempel_ziv_complexity__bins_2:
Aggregated MSE: 0.0011426363328516326
IMPROVEMENT: -0.0006130489299761878
MSE for XGBoost: 0.00040533023135030574
PERCENT IMPROVEMENT: -115.75972665655958%
---------------------------------
Adding all similar features for FEDFUNDS__change_quantiles__f_agg_: ['FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for GDPCTPI__cwt_coefficients__coeff_0__w_5__widths_: ['GDPCTPI__cwt_coefficients__coeff_0__w_5__widths_(2, 5, 10, 20)']
Adding all similar features for HOUST__cwt_coefficients__coeff_0__w_5__widths_: ['HOUST__cwt_coefficients__coeff_0__w_5__widths_(2, 5, 10, 20)']
Adding all similar features for PAYEMS__cwt_coefficients__coeff_0__w_5__widths_: ['PAYEMS__cwt_coefficients__coeff_0__w_5__widths_(2, 5, 10, 20)']
11. ADDING PCEPILFE__index_mass_quantile__q_0.3:
Aggregated MSE: 0.001697796216018117
IMPROVEMENT: -0.0011682088131426721
MSE for XGBoost: 0.00035704914304882855
PERCENT IMPROVEMENT: -220.58848205221122%
---------------------------------
Adding all similar features for PNFI__cwt_coefficients__coeff_0__w_5__widths_: ['PNFI__cwt_coefficients__coeff_0__w_5__widths_(2, 5, 10, 20)']
Target: PCEPI
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 4.6111430012815265e-05
Initial MSE for XGBoost: 2.356969815473325e-05
Initial MSE for LightGBM: 2.2541731858082014e-05
2. ADDING PCEPI__sum_values:
Aggregated MSE: 4.6241805876046714e-05
IMPROVEMENT: -1.3037586323144906e-07
MSE for XGBoost: 2.37000740179647e-05
PERCENT IMPROVEMENT: -0.2827408805044976%
---------------------------------
3. ADDING GCE__last_location_of_maximum:
Aggregated MSE: 4.2553365196320176e-05
IMPROVEMENT: 3.558064816495089e-06
MSE for XGBoost: 2.291418478231267e-05
PERCENT IMPROVEMENT: 7.716231779205793%
---------------------------------
4. ADDING DGS5__quantile__q_0.7:
Aggregated MSE: 4.070867203275575e-05
IMPROVEMENT: 5.4027579800595166e-06
MSE for XGBoost: 2.186504653777853e-05
PERCENT IMPROVEMENT: 11.716743502767068%
---------------------------------
5. ADDING CPIAUCSL__last_location_of_minimum:
Aggregated MSE: 4.025779115476043e-05
IMPROVEMENT: 5.853638858054836e-06
MSE for XGBoost: 2.1931823924200507e-05
PERCENT IMPROVEMENT: 12.69455069258966%
---------------------------------
6. ADDING GDPCTPI__lempel_ziv_complexity__bins_10:
Aggregated MSE: 4.1386949015458516e-05
IMPROVEMENT: 4.724480997356749e-06
MSE for XGBoost: 2.152901656622925e-05
PERCENT IMPROVEMENT: 10.245791544620767%
---------------------------------
Adding all similar features for AAA__change_quantiles__f_agg_: ['AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'AAA__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'AAA__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'AAA__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'AAA__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
8. ADDING BAA__variation_coefficient:
Aggregated MSE: 5.751703850589359e-05
IMPROVEMENT: -1.1405608493078322e-05
MSE for XGBoost: 2.8765760820575262e-05
PERCENT IMPROVEMENT: -24.734883498318055%
---------------------------------
9. ADDING FGCE__percentage_of_reoccurring_datapoints_to_all_datapoints:
Aggregated MSE: 5.104690233694208e-05
IMPROVEMENT: -4.935472324126817e-06
MSE for XGBoost: 2.555169568949696e-05
PERCENT IMPROVEMENT: -10.703359932136465%
---------------------------------
10. ADDING INDPRO__lempel_ziv_complexity__bins_5:
Aggregated MSE: 6.350810325610885e-05
IMPROVEMENT: -1.7396673243293587e-05
MSE for XGBoost: 2.8331701718916017e-05
PERCENT IMPROVEMENT: -37.72746418503766%
---------------------------------
11. ADDING PAYEMS__index_mass_quantile__q_0.9:
Aggregated MSE: 5.717478931867679e-05
IMPROVEMENT: -1.1063359305861525e-05
MSE for XGBoost: 2.571818725097077e-05
PERCENT IMPROVEMENT: -23.992661478481153%
---------------------------------
Adding all similar features for PCEPI__fft_coefficient__attr_: ['PCEPI__fft_coefficient__attr_"real"__coeff_0', 'PCEPI__fft_coefficient__attr_"imag"__coeff_0', 'PCEPI__fft_coefficient__attr_"abs"__coeff_0', 'PCEPI__fft_coefficient__attr_"angle"__coeff_0']
Target: PCEPILFE
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 2.2459499357686667e-05
Initial MSE for XGBoost: 1.1221953582826673e-05
Initial MSE for LightGBM: 1.1237545774859996e-05
2. ADDING HOUST__number_peaks__n_50:
Aggregated MSE: 2.2457565176578076e-05
IMPROVEMENT: 1.9341811085906205e-09
MSE for XGBoost: 1.1220163586945972e-05
PERCENT IMPROVEMENT: 0.008611862080214427%
---------------------------------
3. ADDING DGS2__number_peaks__n_50:
Aggregated MSE: 2.247973690389525e-05
IMPROVEMENT: -2.0237546208584265e-08
MSE for XGBoost: 1.1224688075047883e-05
PERCENT IMPROVEMENT: -0.09010684470870921%
---------------------------------
4. ADDING HOUST__sum_of_reoccurring_values:
Aggregated MSE: 2.2477843948757072e-05
IMPROVEMENT: -1.8344591070405347e-08
MSE for XGBoost: 1.1222444908297971e-05
PERCENT IMPROVEMENT: -0.08167853957139516%
---------------------------------
5. ADDING WTISPLC__minimum:
Aggregated MSE: 2.2478060620562278e-05
IMPROVEMENT: -1.8561262875611416e-08
MSE for XGBoost: 1.1223140027922425e-05
PERCENT IMPROVEMENT: -0.08264326190004277%
---------------------------------
6. ADDING UNRATE__absolute_maximum:
Aggregated MSE: 2.247973378875087e-05
IMPROVEMENT: -2.0234431064204564e-08
MSE for XGBoost: 1.1224234380628747e-05
PERCENT IMPROVEMENT: -0.09009297465608652%
---------------------------------
7. ADDING DGS2__maximum:
Aggregated MSE: 2.2478495500766723e-05
IMPROVEMENT: -1.899614308005672e-08
MSE for XGBoost: 1.1222996092644601e-05
PERCENT IMPROVEMENT: -0.08457954817926684%
---------------------------------
8. ADDING DGS5__symmetry_looking__r_0.05:
Aggregated MSE: 2.2481272405569724e-05
IMPROVEMENT: -2.1773047883056945e-08
MSE for XGBoost: 1.1225772997447601e-05
PERCENT IMPROVEMENT: -0.09694360295526896%
---------------------------------
Adding all similar features for DGS10__cwt_coefficients__coeff_0__w_5__widths_: ['DGS10__cwt_coefficients__coeff_0__w_5__widths_(2, 5, 10, 20)']
Adding all similar features for EXPGS__cwt_coefficients__coeff_0__w_5__widths_: ['EXPGS__cwt_coefficients__coeff_0__w_5__widths_(2, 5, 10, 20)']
Adding all similar features for GDPCTPI__cwt_coefficients__coeff_0__w_5__widths_: ['GDPCTPI__cwt_coefficients__coeff_0__w_5__widths_(2, 5, 10, 20)']
Adding all similar features for HOUST__cwt_coefficients__coeff_0__w_5__widths_: ['HOUST__cwt_coefficients__coeff_0__w_5__widths_(2, 5, 10, 20)']
Adding all similar features for INDPRO__cwt_coefficients__coeff_0__w_5__widths_: ['INDPRO__cwt_coefficients__coeff_0__w_5__widths_(2, 5, 10, 20)']
Adding all similar features for PAYEMS__cwt_coefficients__coeff_0__w_5__widths_: ['PAYEMS__cwt_coefficients__coeff_0__w_5__widths_(2, 5, 10, 20)']
Adding all similar features for PNFI__cwt_coefficients__coeff_0__w_5__widths_: ['PNFI__cwt_coefficients__coeff_0__w_5__widths_(2, 5, 10, 20)']
16. ADDING PRFI__has_duplicate:
Aggregated MSE: 2.2478424854378028e-05
IMPROVEMENT: -1.8925496691361095e-08
MSE for XGBoost: 1.122354670921977e-05
PERCENT IMPROVEMENT: -0.08426499803026075%
---------------------------------
Target: PAYEMS
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.004766032902138348
Initial MSE for XGBoost: 0.002403236891478201
Initial MSE for LightGBM: 0.0023627960106601467
2. ADDING PAYEMS__index_mass_quantile__q_0.8:
Aggregated MSE: 0.004760408555501395
IMPROVEMENT: 5.624346636953596e-06
MSE for XGBoost: 0.0024123739025979666
PERCENT IMPROVEMENT: 0.11800897627941581%
---------------------------------
3. ADDING EXPGS__lempel_ziv_complexity__bins_3:
Aggregated MSE: 0.004795255375323925
IMPROVEMENT: -2.9222473185576414e-05
MSE for XGBoost: 0.0024128298782442087
PERCENT IMPROVEMENT: -0.6131403996910163%
---------------------------------
4. ADDING DGS2__variation_coefficient:
Aggregated MSE: 0.004801100187245611
IMPROVEMENT: -3.5067285107263056e-05
MSE for XGBoost: 0.0024266891901473446
PERCENT IMPROVEMENT: -0.7357751368340244%
---------------------------------
5. ADDING GCE__mean:
Aggregated MSE: 0.0048177408244432845
IMPROVEMENT: -5.170792230493621e-05
MSE for XGBoost: 0.002407681766094934
PERCENT IMPROVEMENT: -1.0849258359449578%
---------------------------------
6. ADDING INDPRO__lempel_ziv_complexity__bins_3:
Aggregated MSE: 0.0048159694626367005
IMPROVEMENT: -4.993656049835215e-05
MSE for XGBoost: 0.0024017879004685438
PERCENT IMPROVEMENT: -1.0477594578910145%
---------------------------------
7. ADDING HOUST__lempel_ziv_complexity__bins_5:
Aggregated MSE: 0.004835581131667665
IMPROVEMENT: -6.954822952931657e-05
MSE for XGBoost: 0.0024042860917796267
PERCENT IMPROVEMENT: -1.4592477844228218%
---------------------------------
8. ADDING DSPI__energy_ratio_by_chunks__num_segments_10__segment_focus_9:
Aggregated MSE: 0.004816934239313163
IMPROVEMENT: -5.0901337174814876e-05
MSE for XGBoost: 0.002403060464065862
PERCENT IMPROVEMENT: -1.0680022194554568%
---------------------------------
Adding all similar features for FEDFUNDS__change_quantiles__f_agg_: ['FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for GDP__change_quantiles__f_agg_: ['GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'GDP__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'GDP__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'GDP__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'GDP__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Adding all similar features for PAYEMS__change_quantiles__f_agg_: ['PAYEMS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'PAYEMS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'PAYEMS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
12. ADDING PAYEMS__last_location_of_minimum:
Aggregated MSE: 0.006848696105362297
IMPROVEMENT: -0.0020826632032239483
MSE for XGBoost: 0.003297607666293398
PERCENT IMPROVEMENT: -43.698045019570294%
---------------------------------
Adding all similar features for PCE__change_quantiles__f_agg_: ['PCE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'PCE__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'PCE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'PCE__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'PCE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'PCE__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'PCE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'PCE__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'PCE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'PCE__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'PCE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'PCE__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'PCE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'PCE__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'PCE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'PCE__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'PCE__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'PCE__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'PCE__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'PCE__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'PCE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'PCE__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'PCE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'PCE__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'PCE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'PCE__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'PCE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'PCE__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'PCE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'PCE__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'PCE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'PCE__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'PCE__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'PCE__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'PCE__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'PCE__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'PCE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'PCE__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'PCE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'PCE__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'PCE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'PCE__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'PCE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'PCE__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'PCE__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'PCE__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'PCE__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'PCE__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'PCE__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'PCE__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'PCE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'PCE__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'PCE__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'PCE__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'PCE__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'PCE__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'PCE__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'PCE__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'PCE__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'PCE__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Target: UNRATE
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.8918977562255643
Initial MSE for XGBoost: 0.4495097315970115
Initial MSE for LightGBM: 0.44238802462855287
2. ADDING UNRATE__sum_values:
Aggregated MSE: 0.8868142566456945
IMPROVEMENT: 0.005083499579869866
MSE for XGBoost: 0.44562048013561695
PERCENT IMPROVEMENT: 0.5699643871045045%
---------------------------------
3. ADDING UNRATE__mean:
Aggregated MSE: 0.884271733070817
IMPROVEMENT: 0.007626023154747363
MSE for XGBoost: 0.4431246396327958
PERCENT IMPROVEMENT: 0.8550333377919961%
---------------------------------
4. ADDING UNRATE__last_location_of_minimum:
Aggregated MSE: 0.8875477720778068
IMPROVEMENT: 0.004349984147757557
MSE for XGBoost: 0.4466810204789004
PERCENT IMPROVEMENT: 0.4877222885015791%
---------------------------------
5. ADDING PCEPI__first_location_of_minimum:
Aggregated MSE: 0.8851164497828751
IMPROVEMENT: 0.006781306442689261
MSE for XGBoost: 0.446506873800723
PERCENT IMPROVEMENT: 0.7603232988708454%
---------------------------------
6. ADDING INDPRO__lempel_ziv_complexity__bins_10:
Aggregated MSE: 0.8962114001092123
IMPROVEMENT: -0.004313643883647966
MSE for XGBoost: 0.45670953952076054
PERCENT IMPROVEMENT: -0.4836478008312232%
---------------------------------
7. ADDING PCEPILFE__energy_ratio_by_chunks__num_segments_10__segment_focus_4:
Aggregated MSE: 0.8937923598988644
IMPROVEMENT: -0.001894603673300077
MSE for XGBoost: 0.4537118341189013
PERCENT IMPROVEMENT: -0.2124238636183904%
---------------------------------
8. ADDING GCE__ratio_beyond_r_sigma__r_2.5:
Aggregated MSE: 0.8936273699522899
IMPROVEMENT: -0.0017296137267255318
MSE for XGBoost: 0.45281106958982154
PERCENT IMPROVEMENT: -0.19392511245292404%
---------------------------------
9. ADDING CPILFESL__count_above__t_0:
Aggregated MSE: 0.8952703994727453
IMPROVEMENT: -0.0033726432471810064
MSE for XGBoost: 0.45343258990580304
PERCENT IMPROVEMENT: -0.37814236257906364%
---------------------------------
10. ADDING CPILFESL__last_location_of_maximum:
Aggregated MSE: 0.8945254195121333
IMPROVEMENT: -0.0026276632865689376
MSE for XGBoost: 0.45498287010531224
PERCENT IMPROVEMENT: -0.29461485559611517%
---------------------------------
Adding all similar features for DSPI__agg_autocorrelation__f_agg_: ['DSPI__agg_autocorrelation__f_agg_"mean"__maxlag_40', 'DSPI__agg_autocorrelation__f_agg_"median"__maxlag_40', 'DSPI__agg_autocorrelation__f_agg_"var"__maxlag_40']
Adding all similar features for FEDFUNDS__change_quantiles__f_agg_: ['FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'FEDFUNDS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'FEDFUNDS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
Target: INDPRO
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.0060985176993022
Initial MSE for XGBoost: 0.003027821289058748
Initial MSE for LightGBM: 0.003070696410243452
2. ADDING INDPRO__last_location_of_maximum:
Aggregated MSE: 0.006130682500281594
IMPROVEMENT: -3.2164800979394247e-05
MSE for XGBoost: 0.003084089914774601
PERCENT IMPROVEMENT: -0.5274199824504008%
---------------------------------
3. ADDING IMPGS__first_location_of_minimum:
Aggregated MSE: 0.006164679738765359
IMPROVEMENT: -6.616203946315912e-05
MSE for XGBoost: 0.003090307750254457
PERCENT IMPROVEMENT: -1.0848872254766018%
---------------------------------
4. ADDING INDPRO__variation_coefficient:
Aggregated MSE: 0.006202410743966061
IMPROVEMENT: -0.00010389304466386102
MSE for XGBoost: 0.003076423404597581
PERCENT IMPROVEMENT: -1.7035786364242018%
---------------------------------
5. ADDING CPILFESL__variation_coefficient:
Aggregated MSE: 0.0061835912679940475
IMPROVEMENT: -8.50735686918477e-05
MSE for XGBoost: 0.0030996303824581692
PERCENT IMPROVEMENT: -1.3949876492378128%
---------------------------------
6. ADDING DSPI__approximate_entropy__m_2__r_0.1:
Aggregated MSE: 0.006177117173331234
IMPROVEMENT: -7.859947402903428e-05
MSE for XGBoost: 0.003064366534714746
PERCENT IMPROVEMENT: -1.2888291533207772%
---------------------------------
7. ADDING PAYEMS__lempel_ziv_complexity__bins_100:
Aggregated MSE: 0.006192640534169165
IMPROVEMENT: -9.412283486696493e-05
MSE for XGBoost: 0.0030938997853621622
PERCENT IMPROVEMENT: -1.5433723325544269%
---------------------------------
8. ADDING IMPGS__energy_ratio_by_chunks__num_segments_10__segment_focus_9:
Aggregated MSE: 0.006231983344918868
IMPROVEMENT: -0.0001334656456166679
MSE for XGBoost: 0.003094707463405776
PERCENT IMPROVEMENT: -2.188493207651742%
---------------------------------
9. ADDING IMPGS__index_mass_quantile__q_0.7:
Aggregated MSE: 0.006238421203816408
IMPROVEMENT: -0.00013990350451420844
MSE for XGBoost: 0.0031461280892412843
PERCENT IMPROVEMENT: -2.294057530245692%
---------------------------------
10. ADDING CUSR0000SAH1__ratio_value_number_to_time_series_length:
Aggregated MSE: 0.0062795194184412165
IMPROVEMENT: -0.00018100171913901664
MSE for XGBoost: 0.00317523070682673
PERCENT IMPROVEMENT: -2.967962512591004%
---------------------------------
11. ADDING AAA__percentage_of_reoccurring_values_to_all_values:
Aggregated MSE: 0.006227437605740268
IMPROVEMENT: -0.00012891990643806774
MSE for XGBoost: 0.003129799541767035
PERCENT IMPROVEMENT: -2.113954780402112%
---------------------------------
12. ADDING INDPRO__abs_energy:
Aggregated MSE: 0.006260260616433526
IMPROVEMENT: -0.00016174291713132605
MSE for XGBoost: 0.0031559671964877718
PERCENT IMPROVEMENT: -2.652167708717692%
---------------------------------
Adding all similar features for INDPRO__change_quantiles__f_agg_: ['INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'INDPRO__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'INDPRO__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'INDPRO__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
14. ADDING INDPRO__last_location_of_minimum:
Aggregated MSE: 0.007365873929579655
IMPROVEMENT: -0.001267356230277455
MSE for XGBoost: 0.0035338611388062327
PERCENT IMPROVEMENT: -20.7813815219798%
---------------------------------
Target: CUMFNS
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 2.8323312670711562
Initial MSE for XGBoost: 1.5056786979299623
Initial MSE for LightGBM: 1.3266525691411937
2. ADDING CUMFNS__mean:
Aggregated MSE: 2.8571037694873547
IMPROVEMENT: -0.02477250241619844
MSE for XGBoost: 1.530451200346161
PERCENT IMPROVEMENT: -0.8746329465131766%
---------------------------------
3. ADDING HOUST__lempel_ziv_complexity__bins_2:
Aggregated MSE: 2.9078793555445417
IMPROVEMENT: -0.07554808847338546
MSE for XGBoost: 1.5301764360057202
PERCENT IMPROVEMENT: -2.6673464842093795%
---------------------------------
4. ADDING DSPI__percentage_of_reoccurring_datapoints_to_all_datapoints:
Aggregated MSE: 2.855986495555478
IMPROVEMENT: -0.02365522848432189
MSE for XGBoost: 1.4960932536456795
PERCENT IMPROVEMENT: -0.8351857976268142%
---------------------------------
5. ADDING PNFI__count_above__t_0:
Aggregated MSE: 2.9043024729874416
IMPROVEMENT: -0.07197120591628536
MSE for XGBoost: 1.5490483878109111
PERCENT IMPROVEMENT: -2.54105890624542%
---------------------------------
6. ADDING PNFI__count_below__t_0:
Aggregated MSE: 2.872852998547347
IMPROVEMENT: -0.04052173147619076
MSE for XGBoost: 1.5076957483374955
PERCENT IMPROVEMENT: -1.4306847488956784%
---------------------------------
7. ADDING AAA__number_peaks__n_1:
Aggregated MSE: 2.8785729604179098
IMPROVEMENT: -0.04624169334675354
MSE for XGBoost: 1.4828019591088788
PERCENT IMPROVEMENT: -1.6326371806985323%
---------------------------------
Adding all similar features for CUMFNS__change_quantiles__f_agg_: ['CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.2__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.2__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.0', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.4__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.4__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.4__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.4__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.2', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.6__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.6__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.6__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.4', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_0.8__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_0.8__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.6', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_False__qh_1.0__ql_0.8', 'CUMFNS__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8', 'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_1.0__ql_0.8']
9. ADDING CUMFNS__last_location_of_maximum:
Aggregated MSE: 3.3859333058288335
IMPROVEMENT: -0.5536020387576772
MSE for XGBoost: 1.7209401791239503
PERCENT IMPROVEMENT: -19.545808260279646%
---------------------------------
Target: USREC
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 1.201110230730729
Initial MSE for XGBoost: 0.6005551153653645
Initial MSE for LightGBM: 0.6005551153653645
2. ADDING FEDFUNDS__has_duplicate_max:
Aggregated MSE: 1.201110230730729
IMPROVEMENT: 0.0
MSE for XGBoost: 0.6005551153653645
PERCENT IMPROVEMENT: 0.0%
---------------------------------
3. ADDING FEDFUNDS__has_duplicate_min:
Aggregated MSE: 1.201110230730729
IMPROVEMENT: 0.0
MSE for XGBoost: 0.6005551153653645
PERCENT IMPROVEMENT: 0.0%
---------------------------------
4. ADDING FEDFUNDS__variance_larger_than_standard_deviation:
Aggregated MSE: 1.201110230730729
IMPROVEMENT: 0.0
MSE for XGBoost: 0.6005551153653645
PERCENT IMPROVEMENT: 0.0%
---------------------------------
"""
import re

# Define the regular expression pattern to match the "Adding all similar features for" sections
pattern = r"Adding all similar features for.*?: \[.*?\]"

# Use re.sub() to remove the matched sections from the text
cleaned_text = re.sub(pattern, '', text, flags=re.DOTALL)

# Print the cleaned text
print(cleaned_text)


Target: EXPGS
1. BASE FEATURE EVAL:
Initial aggregated baseline MSE: 0.0007083075575865404
Initial MSE for XGBoost: 0.0003575527559418474
Initial MSE for LightGBM: 0.00035075480164469306
2. ADDING PNFI__longest_strike_below_mean:
Aggregated MSE: 0.0007900623185914516
IMPROVEMENT: -8.175476100491118e-05
MSE for XGBoost: 0.00039210810690614204
PERCENT IMPROVEMENT: -11.542268627413657%
---------------------------------
3. ADDING DGS2__number_peaks__n_10:
Aggregated MSE: 0.0007621176066378139
IMPROVEMENT: -5.381004905127341e-05
MSE for XGBoost: 0.0003613206884182433
PERCENT IMPROVEMENT: -7.596989256280658%
---------------------------------
4. ADDING DGS10__energy_ratio_by_chunks__num_segments_10__segment_focus_8:
Aggregated MSE: 0.0007937651825563744
IMPROVEMENT: -8.545762496983396e-05
MSE for XGBoost: 0.00034976976947832344
PERCENT IMPROVEMENT: -12.065044916507588%
---------------------------------
5. ADDING PAYEMS__cid_ce__normalize_True:
Aggregated MSE: 0.0007386971039901024
IMPROVEMEN

In [55]:
import re
import ast
import pandas as pd
# Load the processed data
processed_train_path = 'data/processed/train_transformed_combined.csv'
processed_test_path = 'data/processed/test_transformed_combined.csv'
tsfresh_train_path = 'data/tsfresh/train_combined_all_features_filled.csv'
tsfresh_test_path = 'data/tsfresh/test_combined_all_features_filled.csv'

train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)


# EXAMPLE INPUTS:
target = 'INDPRO'
features_to_add = ['INDPRO__last_location_of_maximum', 'IMPGS__first_location_of_minimum', 'INDPRO__variation_coefficient', 'CPILFESL__variation_coefficient', 'DSPI__approximate_entropy__m_2__r_0.1', 'PAYEMS__lempel_ziv_complexity__bins_100', 'IMPGS__energy_ratio_by_chunks__num_segments_10__segment_focus_9', 'IMPGS__index_mass_quantile__q_0.7', 'CUSR0000SAH1__ratio_value_number_to_time_series_length', 'AAA__percentage_of_reoccurring_values_to_all_values', 'INDPRO__abs_energy']

def compute_mse_scores(X_train_with_feature, X_test_with_feature, y_train, y_test):
    xgboost_params = {'max_depth': 9, 'learning_rate': 0.016810144010995204, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5865511380196138, 'colsample_bytree': 0.9177433017782509, 'reg_alpha': 0.012513778476694921, 'reg_lambda': 8.577576991968611e-05}
    lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832469304666387, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402007213047204, 'colsample_bytree': 0.9405245932820381, 'reg_alpha': 0.000756445935013929, 'reg_lambda': 0.0002590470172405959}
    xgboost = XGBRegressor(**xgboost_params)
    lightgbm = LGBMRegressor(**lightgbm_params,verbosity=-1)
    models = {'XGBoost': xgboost, 'LightGBM': lightgbm}
    mse_scores = {}

    for model_name, model in models.items():
        model.fit(X_train_with_feature, y_train)
        y_pred = model.predict(X_test_with_feature)
        mse = mean_squared_error(y_test, y_pred)
        mse_scores[model_name] = mse
    
    aggregated_mse = np.mean(list(mse_scores.values()))

    # calculate train_mse_scores
    train_mse_scores = {}
    for model_name, model in models.items():
        y_pred = model.predict(X_train_with_feature)
        mse = mean_squared_error(y_train, y_pred)
        train_mse_scores[model_name] = mse


    return mse_scores, train_mse_scores, aggregated_mse


def clean_feature_names(df):
    df.columns = df.columns.str.replace(r'[^A-Za-z0-9]+', '', regex=True)
    return df

def eval_feature(target, train_combined, test_combined, tsfresh_features_test, tsfresh_features_train, features_to_add):
    # Extract base features for target
    feature_engineering_results = pd.read_csv('feature_engineering_results.csv')
    base_features_str = feature_engineering_results.loc[feature_engineering_results['target'] == target, 'final_feature_space'].values[0]
    base_features = [re.sub(r'""(.*?)""', r'"\1"', feature) for feature in ast.literal_eval(base_features_str)]
    valid_base_features = [feature for feature in base_features if feature in tsfresh_features_train.columns]

    # Clean feature names
    X_train_base = clean_feature_names(tsfresh_features_train[valid_base_features])
    X_test_base = clean_feature_names(tsfresh_features_test[valid_base_features])
    y_train = train_combined[[target]].values.ravel()
    y_test = test_combined[[target]].values.ravel()

    # Initial MSE evaluation
    baseline_mse_scores, baseline_train_mse_scores, baseline_aggregated_mse = compute_mse_scores(X_train_base, X_test_base, y_train, y_test)
    print("Baseline MSE (Aggregated):", baseline_aggregated_mse)
    print("Baseline MSE (XGBoost):", baseline_mse_scores['XGBoost'])
    print("Baseline MSE (LightGBM):", baseline_mse_scores['LightGBM'])

    # Evaluate the effect of each feature in isolation
    for feature in features_to_add:
        if feature in tsfresh_features_train.columns:
            # Evaluate the effect of adding the feature individually
            X_train_feature_to_add = clean_feature_names(tsfresh_features_train[[feature]].copy())
            X_test_feature_to_add = clean_feature_names(tsfresh_features_test[[feature]].copy())
            X_train_with_feature = pd.concat([X_train_base, X_train_feature_to_add], axis=1)
            X_test_with_feature = pd.concat([X_test_base, X_test_feature_to_add], axis=1)

            # Compute the MSE for the model with the additional feature
            mse_scores, train_mse_scores, aggregated_mse = compute_mse_scores(X_train_with_feature, X_test_with_feature, y_train, y_test)

            print(f"\nFeature: {feature}")
            print(f"Aggregated MSE: {aggregated_mse}")
            print(f"MSE for XGBoost: {mse_scores['XGBoost']}")
            print(f"MSE for LightGBM: {mse_scores['LightGBM']}")
            print(f"Change in MSE: {aggregated_mse - baseline_aggregated_mse}")
            print(f"Percentage Change in MSE: {(aggregated_mse - baseline_aggregated_mse) / baseline_aggregated_mse * 100}%")

            # Remove the added feature
            X_train_base.drop(columns=[feature], inplace=True)
            X_test_base.drop(columns=[feature], inplace=True)



# Example use: PCEPI with features: [DGS5__quantile__q_0.7,CPIAUCSL__last_location_of_minimum]
target = 'INDPRO'
features_to_add = ['CPIAUCSL__last_location_of_minimum']
eval_feature(target, train_combined, test_combined, tsfresh_features_test, tsfresh_features_train,features_to_add)


Baseline MSE (Aggregated): 0.003052718558039481
Baseline MSE (XGBoost): 0.003021205429902479
Baseline MSE (LightGBM): 0.003084231686176484

Feature: CPIAUCSL__last_location_of_minimum
Aggregated MSE: 0.003047731691947199
MSE for XGBoost: 0.0030227156984105295
MSE for LightGBM: 0.0030727476854838683
Change in MSE: -4.98686609228232e-06
Percentage Change in MSE: -0.16335820015733743%


KeyError: "['CPIAUCSL__last_location_of_minimum'] not found in axis"

In [279]:
import ast
import os
import pandas as pd
import numpy as np
import mlflow
import dagshub
import optuna
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.metrics import mean_squared_error, mean_absolute_error
from funcs.api_funcs import get_target_arg, get_feature_addition_rounds_arg, get_feature_dropping_threshold_arg, get_tsfresh_fc_params_arg
from autogluon.tabular import TabularPredictor
from funcs.train_model_funcs import clean_data
from best_params import xgboost_params, lightgbm_params

def train_model_with_features(X_train, y_train, X_test, y_test, xgboost_params, lightgbm_params):
        # Train the model with the base features only
    xgboost = xgb.XGBRegressor(**xgboost_params)
    lgbm = lgb.LGBMRegressor(**lightgbm_params, verbosity=-1)
    models = {'XGBoost': xgboost, 'LightGBM': lgbm}
    mse_scores = {}
    rmse_scores = {}
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mse_scores[model_name] = mse
        rmse_scores[model_name] = rmse
    # Calculate the aggregated MSE and RMSE
    aggregated_mse = np.mean(list(mse_scores.values()))
    aggregated_rmse = np.mean(list(rmse_scores.values()))
    print(f"Aggregated MSE for {target}: {aggregated_mse}")
    print(f"Aggregated RMSE for {target}: {aggregated_rmse}")
    print(f"MSE for XGBoost: {mse_scores['XGBoost']}")
    print(f"RMSE for XGBoost: {rmse_scores['XGBoost']}")
    xgboost_mse = mse_scores['XGBoost']
    xgboost_rmse = rmse_scores['XGBoost']
    return xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse
    

targets = ['FEDFUNDS', 'GDP', 'CPIAUCSL', 'CUSR0000SAH1', 'CPILFESL', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'HOUST', 'DSPI', 
           'DGS2', 'DGS5', 'DGS10', 'AAA', 'BAA', 'WTISPLC', 'IMPGS', 'GCE', 'FGCE', 'GDPCTPI', 'PCEPI', 'PCEPILFE', 
           'PAYEMS', 'UNRATE', 'INDPRO', 'CUMFNS', 'USREC']

for target in targets:
    feature_space = pd.read_csv('feature_engineering_results_round2.csv')
    top_lagged_features = pd.read_csv(r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\top_features_combined_rmse_mse_results.csv')
    feature_space_mask = (feature_space==target).any(axis=1)
    feature_space_list = feature_space[feature_space_mask]['final_feature_space'].values.tolist()
    feature_space_series =pd.Series(feature_space_list)
    feature_space_series
    # Initialize 'results' as an empty DataFrame
    results = pd.DataFrame(columns=['target', 'combination', 'xgboost_mse', 'xgboost_rmse', 'aggregated_mse', 'aggregated_rmse'])
    # Example where feature_space is a pandas Series that contains string representations of lists
    feature_space = feature_space_series.apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    # Ensure both feature_space and top_lagged_features are lists
    feature_space_list = feature_space.tolist()  # May contain lists as elements
    # top_lagged_features_list = top_lagged_features.tolist()
    feature_space_list
    # # Flatten the feature_space_list to remove any nested lists
    flat_feature_space = [item for sublist in feature_space_list for item in sublist] if any(isinstance(i, list) for i in feature_space_list) else feature_space_list
    top_lagged_features_rows
    top_lagged_features_list = []
    for feature in top_lagged_features_rows['feature']:
        top_lagged_features_list.append(feature)
    top_lagged_features_list
    # Combine the flattened lists
    features_to_add = flat_feature_space + top_lagged_features_list
    # Train First using the base features only
    X_train = pd.read_csv('data/processed/X_train_transformed.csv',index_col='Date', parse_dates=True)
    y_train = pd.read_csv('data/processed/y_train_transformed.csv',index_col='Date', parse_dates=True)
    X_test = pd.read_csv('data/processed/X_test_transformed.csv',index_col='Date', parse_dates=True)
    y_test = pd.read_csv('data/processed/y_test_transformed.csv',index_col='Date', parse_dates=True)
    
    # display(X_train.head())
    # display(y_train.head())

    
    print("Evaluation for target:", target)
    print("1. Evaluation base feature performance for target:", target)
    # search dataframes for categorical types/columns
    categorical_columns = (X_train.dtypes == 'object')
    # display(f"Categorical columns: {categorical_columns[categorical_columns].index.tolist()}")
    xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse = train_model_with_features(X_train, y_train, X_test, y_test, xgboost_params, lightgbm_params)
    # Save results of base feature evaluation in results, ranked by aggregated MSE
    results = results.append({'target': target, 'combination': 'Base Features','xgboost_mse': xgboost_mse, 'xgboost_rmse': xgboost_rmse, 'aggregated_mse': aggregated_mse, 'aggregated_rmse': aggregated_rmse},ignore_index=True)

    # Evaluate with added feature space 
    tsfresh_train_combined = pd.read_csv('data/tsfresh/train_combined_all_features_filled.csv',index_col='Date', parse_dates=True)
    tsfresh_test_combined = pd.read_csv('data/tsfresh/test_combined_all_features_filled.csv',index_col='Date', parse_dates=True)
    X_train_tsfresh = tsfresh_train_combined[flat_feature_space]
    X_test_tsfresh = tsfresh_test_combined[flat_feature_space]
    # display(f"X_train_tsfresh features: {X_train_tsfresh.columns}")
    # display(f"X_train tsfresh NAN values: {X_train_tsfresh.isnull().sum().sum()}")
    # display(f"X_train_tsfresh shape: {X_train_tsfresh.shape}")
    # display(f"X_train regular shape: {X_train.shape}")
    # display(f"X_train regular NAN values: {X_train.isnull().sum().sum()}")

    print("2. Evaluation engineered feature performance only for target:", target)
    import re
    X_train_tsfresh = X_train.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    X_test_tsfresh = X_test.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse = train_model_with_features(X_train_tsfresh, y_train, X_test_tsfresh, y_test, xgboost_params, lightgbm_params)
    # Save results of base feature evaluation in results, ranked by aggregated MSE
    results = results.append({'target': target, 'combination': 'Engineered Features','xgboost_mse': xgboost_mse, 'xgboost_rmse': xgboost_rmse, 'aggregated_mse': aggregated_mse, 'aggregated_rmse': aggregated_rmse},ignore_index=True)

    X_train = pd.concat([X_train, X_train_tsfresh], axis=1)
    # display(f"X_train shape AFTER addition: {X_train.shape}")
    # display(f"X_train NAN after addition: {X_train.isnull().sum().sum()}")
    X_test = pd.concat([X_test, X_test_tsfresh], axis=1)
    # X_train, X_test = clean_data(X_train, X_test)
    # display(f"X_train shape AFTER cleaning data: {X_train.shape}")
    # Evaluate the effect of each feature, all together
    
    print("3. Evaluating feature space performance with engineered features + base_features:", target)
    # display("DEBUGGING")
    # display(f"Nan X_train: {X_train.isnull().sum().sum()}")
    # display(f"Nan X_test: {X_test.isnull().sum().sum()}")
    # display(f"Nan y_train: {y_train.isnull().sum().sum()}")
    # display(f"Nan y_test: {y_test.isnull().sum().sum()}")
    # display(f"SHape X_train: {X_train.shape}")
    # display(f"SHape X_test: {X_test.shape}")
    # display(f"SHape y_train: {y_train.shape}")
    # display(f"SHape y_test: {y_test.shape}")
    # print(f"Whether X_train indexes match: {X_train.index.equals(X_train_tsfresh.index)}")  # Should return True
    # print(f"should return False, column overlap: {X_train.columns.equals(X_train_tsfresh.columns)}")
    # categorical_columns = (X_train.dtypes == 'object')
    # display(f"Categorical columns: {categorical_columns[categorical_columns].index.tolist()}")
    # display("REMOVING SPECIAL DATA TYPES AND OBJECTS")
    # X_train = X_train.apply(pd.to_numeric, errors='coerce')
    # X_test = X_test.apply(pd.to_numeric, errors='coerce')
    # display("TRYING TO CONVERT ALL SPECIAL CHARACTERS")
    import re
    X_train = X_train.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    X_test = X_test.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse = train_model_with_features(X_train, y_train, X_test, y_test, xgboost_params, lightgbm_params)
    # Save results of base feature evaluation in results, ranked by aggregated MSE
    results = results.append({'target': target, 'combination': 'Engineered + Base Features','xgboost_mse': xgboost_mse, 'xgboost_rmse': xgboost_rmse, 'aggregated_mse': aggregated_mse, 'aggregated_rmse': aggregated_rmse}, ignore_index=True)

    # Evaluate with added top lagged features
    lagged_train_combined = pd.read_csv('data/engineered/combined_lag_train.csv',index_col='Date', parse_dates=True)
    lagged_test_combined = pd.read_csv('data/engineered/combined_lag_test.csv',index_col='Date', parse_dates=True)
    X_train_lagged = lagged_train_combined[top_lagged_features_list]
    X_test_lagged = lagged_test_combined[top_lagged_features_list]
    X_train = X_train + X_train_lagged
    X_test = X_test + X_test_lagged
    # Evaluate the effect of each feature, all together
    
    print("4. Evaluating feature space performance with top lagged features + engineered features + base features:", target)
    X_train = X_train.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    X_test = X_test.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse = train_model_with_features(X_train, y_train, X_test, y_test, xgboost_params, lightgbm_params)
    # Save results of base feature evaluation in results, ranked by aggregated MSE
    results = results.append({'target': target, 'combination': 'Lagged + Base + Engineered Features','xgboost_mse': xgboost_mse, 'xgboost_rmse': xgboost_rmse, 'aggregated_mse': aggregated_mse, 'aggregated_rmse': aggregated_rmse}, ignore_index=True)

    # Evaluate with added top lagged features
    # X_train = pd.concat([X_train,X_train_lagged], axis=1)
    # X_train, X_test = clean_data(X_train, X_test)
    # Evaluate the effect of each feature, all together
    
    # Evaluate the effect of only the lagged, and engineered features
    print("5. Evaluating feature space performance with top lagged features + engineered features only:", target)
    tsfresh_train_combined = pd.read_csv(r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\train_combined_all_features_filled.csv',index_col='Date', parse_dates=True)
    tsfresh_test_combined = pd.read_csv(r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\test_combined_all_features_filled.csv',index_col='Date', parse_dates=True)
    X_train_tsfresh = tsfresh_train_combined[flat_feature_space]
    X_test_tsfresh = tsfresh_test_combined[flat_feature_space]
    X_train =X_train_tsfresh
    X_test = X_test_tsfresh
    # X_train, X_test = clean_data(X_train, X_test)
    lagged_train_combined = pd.read_csv('data/engineered/combined_lag_train.csv',index_col='Date', parse_dates=True)
    lagged_test_combined = pd.read_csv('data/engineered/combined_lag_test.csv',index_col='Date', parse_dates=True)
    X_train_lagged = lagged_train_combined[top_lagged_features_list]
    X_test_lagged = lagged_test_combined[top_lagged_features_list]
    X_train = pd.concat([X_train,X_train_lagged], axis=1)
    X_test = pd.concat([X_test,X_test_lagged], axis=1)
    X_train, X_test = clean_data(X_train, X_test)
    print("4. Evaluating feature space performance with top lagged features + engineered features + base features:", target)
    X_train = X_train.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    X_test = X_test.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse = train_model_with_features(X_train, y_train, X_test, y_test, xgboost_params, lightgbm_params)
    # Save results of base feature evaluation in results, ranked by aggregated MSE
    results = results.append({'target': target, 'combination': 'Lagged + Engineered Features','xgboost_mse': xgboost_mse, 'xgboost_rmse': xgboost_rmse, 'aggregated_mse': aggregated_mse, 'aggregated_rmse': aggregated_rmse}, ignore_index=True)
    # 5. Evaluate effect of adding only the top lagged feature to the base features
    print("6. Evaluating feature space performance with top lagged feature + base feature only:", target)
    X_train = pd.read_csv('data/processed/X_train_transformed.csv',index_col='Date', parse_dates=True)
    y_train = pd.read_csv('data/processed/y_train_transformed.csv',index_col='Date', parse_dates=True)
    X_test = pd.read_csv('data/processed/X_test_transformed.csv',index_col='Date', parse_dates=True)
    y_test = pd.read_csv('data/processed/y_test_transformed.csv',index_col='Date', parse_dates=True)
    # Sort the top lagged features by 'pct_improvement_mse' in descending order and select the top one
    best_lagged_feature = top_lagged_features.sort_values(by='pct_improvement_mse', ascending=False).iloc[0]['feature']
    X_train_lagged = lagged_train_combined[[best_lagged_feature]]
    X_test_lagged = lagged_test_combined[[best_lagged_feature]]
    X_train = X_train + X_train_lagged
    X_test = X_test + X_test_lagged
    X_train = X_train.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    X_test = X_test.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse = train_model_with_features(X_train, y_train, X_test, y_test, xgboost_params, lightgbm_params)
    # Save results of base feature evaluation in results, ranked by aggregated MSE
    results = results.append({'target': target, 'combination': 'Top_Lagged + Base Features','xgboost_mse': xgboost_mse, 'xgboost_rmse': xgboost_rmse, 'aggregated_mse': aggregated_mse, 'aggregated_rmse': aggregated_rmse}, ignore_index=True)
    # Now `best_lagged_feature` contains the name of the best lagged feature
    # 6. Evaluate effect of adding only the top lagged feature to the engineered features
    print("7. Evaluating feature space performance with top lagged feature + engineered features only:", target)
    tsfresh_train_combined = pd.read_csv(r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\train_combined_all_features_filled.csv',index_col='Date', parse_dates=True)
    tsfresh_test_combined = pd.read_csv(r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\tsfresh\test_combined_all_features_filled.csv',index_col='Date', parse_dates=True)
    X_train_tsfresh = tsfresh_train_combined[flat_feature_space]
    X_test_tsfresh = tsfresh_test_combined[flat_feature_space]
    X_train = pd.concat([X_train_tsfresh, X_train_lagged], axis=1)
    X_test = pd.concat([X_test_tsfresh, X_test_lagged], axis=1)
    X_train = X_train.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    X_test = X_test.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse = train_model_with_features(X_train, y_train, X_test, y_test, xgboost_params, lightgbm_params)
    # Save results of base feature evaluation in results, ranked by aggregated MSE
    results = results.append({'target': target, 'combination': 'Top_Lagged + Engineered Features','xgboost_mse': xgboost_mse, 'xgboost_rmse': xgboost_rmse, 'aggregated_mse': aggregated_mse, 'aggregated_rmse': aggregated_rmse}, ignore_index=True)
    # 7. Evaluate effect of adding  the top lagged feature + engineered features + base features
    print("8. Evaluating feature space performance with top lagged feature + engineered features + base features:", target)
    X_train_t = pd.read_csv('data/processed/X_train_transformed.csv',index_col='Date', parse_dates=True)
    y_train_t = pd.read_csv('data/processed/y_train_transformed.csv',index_col='Date', parse_dates=True)
    X_test_t = pd.read_csv('data/processed/X_test_transformed.csv',index_col='Date', parse_dates=True)
    y_test_t = pd.read_csv('data/processed/y_test_transformed.csv',index_col='Date', parse_dates=True)
    X_train = pd.concat([X_train_t, X_train], axis=1)
    X_test = pd.concat([X_test_t, X_test], axis=1)
    X_train = X_train.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    X_test = X_test.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse = train_model_with_features(X_train, y_train, X_test, y_test, xgboost_params, lightgbm_params)
    # Save results of base feature evaluation in results, ranked by aggregated MSE
    results = results.append({'target': target, 'combination': 'Top_Lagged + Engineered + Base Features','xgboost_mse': xgboost_mse, 'xgboost_rmse': xgboost_rmse, 'aggregated_mse': aggregated_mse, 'aggregated_rmse': aggregated_rmse}, ignore_index=True)

    # Rank and display the results stored in 'results' first in terms of mse, then RMSE
    print("RESULTS FOR TARGET:", target)
    results.sort_values(by='aggregated_mse', ascending=True, inplace=True)
    display(results)
    results.sort_values(by='aggregated_rmse', ascending=True, inplace=True)
    display(results)




    

Evaluation for target: FEDFUNDS
1. Evaluation base feature performance for target: FEDFUNDS
Aggregated MSE for FEDFUNDS: 0.0004543434866931109
Aggregated RMSE for FEDFUNDS: 0.021305157053098794
MSE for XGBoost: 0.0004824071466612597
RMSE for XGBoost: 0.021963768953921815


AttributeError: 'DataFrame' object has no attribute 'append'

In [286]:
import ast
import os
import pandas as pd
import numpy as np
import mlflow
import dagshub
import optuna
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.metrics import mean_squared_error, mean_absolute_error
from funcs.api_funcs import get_target_arg, get_feature_addition_rounds_arg, get_feature_dropping_threshold_arg, get_tsfresh_fc_params_arg
from autogluon.tabular import TabularPredictor
from funcs.train_model_funcs import clean_data
from best_params import xgboost_params, lightgbm_params

def train_model_with_features(X_train, y_train, X_test, y_test, xgboost_params, lightgbm_params):
    xgboost = xgb.XGBRegressor(**xgboost_params)
    lgbm = lgb.LGBMRegressor(**lightgbm_params, verbosity=-1)
    models = {'XGBoost': xgboost, 'LightGBM': lgbm}
    mse_scores, rmse_scores = {}, {}
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mse, rmse = mean_squared_error(y_test, y_pred), np.sqrt(mean_squared_error(y_test, y_pred))
        mse_scores[model_name], rmse_scores[model_name] = mse, rmse
    return mse_scores['XGBoost'], rmse_scores['XGBoost'], np.mean(list(mse_scores.values())), np.mean(list(rmse_scores.values()))

targets = ['FEDFUNDS', 'GDP', 'CPIAUCSL', 'CUSR0000SAH1', 'CPILFESL', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'HOUST', 'DSPI', 
            'DGS5', 'DGS10', 'AAA', 'BAA', 'WTISPLC', 'IMPGS', 'FGCE',  'PCEPI', 'PCEPILFE', 
           'PAYEMS', 'UNRATE', 'INDPRO', 'CUMFNS', 'USREC']

for target in targets:
    feature_space = pd.read_csv('feature_engineering_results_round2.csv')
    top_lagged_features = pd.read_csv(r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\top_features_combined_rmse_mse_results.csv')
    feature_space_mask = (feature_space == target).any(axis=1)
    feature_space_list = feature_space[feature_space_mask]['final_feature_space'].values.tolist()
    feature_space_series = pd.Series(feature_space_list)
    feature_space = feature_space_series.apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    flat_feature_space = [item for sublist in feature_space.tolist() for item in sublist] if any(isinstance(i, list) for i in feature_space.tolist()) else feature_space.tolist()
    top_lagged_features_list = top_lagged_features.loc[top_lagged_features['target'] == target]['feature'].tolist()
    features_to_add = flat_feature_space + top_lagged_features_list

    X_train = pd.read_csv('data/processed/X_train_transformed.csv', index_col='Date', parse_dates=True)
    y_train = pd.read_csv('data/processed/y_train_transformed.csv', index_col='Date', parse_dates=True)
    X_test = pd.read_csv('data/processed/X_test_transformed.csv', index_col='Date', parse_dates=True)
    y_test = pd.read_csv('data/processed/y_test_transformed.csv', index_col='Date', parse_dates=True)
    
    results = pd.DataFrame(columns=['target', 'combination', 'xgboost_mse', 'xgboost_rmse', 'aggregated_mse', 'aggregated_rmse'])

    xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse = train_model_with_features(X_train, y_train, X_test, y_test, xgboost_params, lightgbm_params)
    results_row_df = pd.DataFrame({'target': target, 'combination': 'Base Features', 'xgboost_mse': xgboost_mse, 'xgboost_rmse': xgboost_rmse, 'aggregated_mse': aggregated_mse, 'aggregated_rmse': aggregated_rmse}, index=[0])
    results = pd.concat([results, results_row_df], ignore_index=True)

    tsfresh_train_combined = pd.read_csv('data/tsfresh/train_combined_all_features_filled.csv', index_col='Date', parse_dates=True)
    tsfresh_test_combined = pd.read_csv('data/tsfresh/test_combined_all_features_filled.csv', index_col='Date', parse_dates=True)
    X_train_tsfresh = tsfresh_train_combined[flat_feature_space]
    X_test_tsfresh = tsfresh_test_combined[flat_feature_space]

    X_train_tsfresh = X_train_tsfresh.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x))
    X_test_tsfresh = X_test_tsfresh.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x))
    xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse = train_model_with_features(X_train_tsfresh, y_train, X_test_tsfresh, y_test, xgboost_params, lightgbm_params)
    results_row_df = pd.DataFrame({'target': target, 'combination': 'Engineered Features', 'xgboost_mse': xgboost_mse, 'xgboost_rmse': xgboost_rmse, 'aggregated_mse': aggregated_mse, 'aggregated_rmse': aggregated_rmse}, index=[0])
    results = pd.concat([results, results_row_df], ignore_index=True)

    X_train = pd.concat([X_train, X_train_tsfresh], axis=1)
    X_test = pd.concat([X_test, X_test_tsfresh], axis=1)
    xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse = train_model_with_features(X_train, y_train, X_test, y_test, xgboost_params, lightgbm_params)
    results_row_df = pd.DataFrame({'target': target, 'combination': 'Engineered + Base Features', 'xgboost_mse': xgboost_mse, 'xgboost_rmse': xgboost_rmse, 'aggregated_mse': aggregated_mse, 'aggregated_rmse': aggregated_rmse}, index=[0])
    results = pd.concat([results, results_row_df], ignore_index=True)

    lagged_train_combined = pd.read_csv('data/engineered/combined_lag_train.csv', index_col='Date', parse_dates=True)
    lagged_test_combined = pd.read_csv('data/engineered/combined_lag_test.csv', index_col='Date', parse_dates=True)
    X_train_lagged = lagged_train_combined[top_lagged_features_list]
    X_test_lagged = lagged_test_combined[top_lagged_features_list]
    X_train = X_train + X_train_lagged
    X_test = X_test + X_test_lagged
    xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse = train_model_with_features(X_train, y_train, X_test, y_test, xgboost_params, lightgbm_params)
    results_row_df = pd.DataFrame({'target': target, 'combination': 'Lagged + Base + Engineered Features', 'xgboost_mse': xgboost_mse, 'xgboost_rmse': xgboost_rmse, 'aggregated_mse': aggregated_mse, 'aggregated_rmse': aggregated_rmse}, index=[0])
    results = pd.concat([results, results_row_df], ignore_index=True)

    best_lagged_feature = top_lagged_features.sort_values(by='pct_improvement_mse', ascending=False).iloc[0]['feature']
    X_train_lagged = lagged_train_combined[[best_lagged_feature]]
    X_test_lagged = lagged_test_combined[[best_lagged_feature]]
    X_train = X_train + X_train_lagged
    X_test = X_test + X_test_lagged
    xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse = train_model_with_features(X_train, y_train, X_test, y_test, xgboost_params, lightgbm_params)
    results_row_df = pd.DataFrame({'target': target, 'combination': 'Top_Lagged + Base Features', 'xgboost_mse': xgboost_mse, 'xgboost_rmse': xgboost_rmse, 'aggregated_mse': aggregated_mse, 'aggregated_rmse': aggregated_rmse}, index=[0])
    results = pd.concat([results, results_row_df], ignore_index=True)

    X_train = pd.concat([X_train_tsfresh, X_train_lagged], axis=1)
    X_test = pd.concat([X_test_tsfresh, X_test_lagged], axis=1)
    xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse = train_model_with_features(X_train, y_train, X_test, y_test, xgboost_params, lightgbm_params)
    results_row_df = pd.DataFrame({'target': target, 'combination': 'Top_Lagged + Engineered Features', 'xgboost_mse': xgboost_mse, 'xgboost_rmse': xgboost_rmse, 'aggregated_mse': aggregated_mse, 'aggregated_rmse': aggregated_rmse}, index=[0])
    results = pd.concat([results, results_row_df], ignore_index=True)

    X_train_t = pd.read_csv('data/processed/X_train_transformed.csv',index_col='Date', parse_dates=True)
    y_train_t = pd.read_csv('data/processed/y_train_transformed.csv',index_col='Date', parse_dates=True)
    X_test_t = pd.read_csv('data/processed/X_test_transformed.csv',index_col='Date', parse_dates=True)
    y_test_t = pd.read_csv('data/processed/y_test_transformed.csv',index_col='Date', parse_dates=True)
    X_train = pd.concat([X_train_t, X_train], axis=1)
    X_test = pd.concat([X_test_t, X_test], axis=1)
    xgboost_mse, xgboost_rmse, aggregated_mse, aggregated_rmse = train_model_with_features(X_train, y_train, X_test, y_test, xgboost_params, lightgbm_params)
    results_row_df = pd.DataFrame({'target': target, 'combination': 'Top_Lagged + Engineered + Base Features', 'xgboost_mse': xgboost_mse, 'xgboost_rmse': xgboost_rmse, 'aggregated_mse': aggregated_mse, 'aggregated_rmse': aggregated_mse}, index=[0])
    results = pd.concat([results, results_row_df], ignore_index=True)

    print("RESULTS FOR TARGET:", target)
    # sort the results DF by aggregated MSE, and then RMSE, ranking best combination
    results.sort_values(by='aggregated_mse', ascending=True, inplace=True)
    display(results)
    results.sort_values(by='aggregated_rmse', ascending=True, inplace=True)
    display(results)


RESULTS FOR TARGET: FEDFUNDS


,target,combination,xgboost_mse,xgboost_rmse,aggregated_mse,aggregated_rmse
6,FEDFUNDS,Top_Lagged + Engineered + Base Features,0.000459,0.021418,0.000435,0.000435
2,FEDFUNDS,Engineered + Base Features,0.000477,0.021840,0.000442,0.020998
0,FEDFUNDS,Base Features,0.000482,0.021964,0.000454,0.021305
3,FEDFUNDS,Lagged + Base + Engineered Features,0.000753,0.027438,0.000753,0.027436
4,FEDFUNDS,Top_Lagged + Base Features,0.000753,0.027438,0.000753,0.027436
1,FEDFUNDS,Engineered Features,0.000797,0.028230,0.000795,0.028201
5,FEDFUNDS,Top_Lagged + Engineered Features,0.000875,0.029587,0.000814,0.028504


,target,combination,xgboost_mse,xgboost_rmse,aggregated_mse,aggregated_rmse
6,FEDFUNDS,Top_Lagged + Engineered + Base Features,0.000459,0.021418,0.000435,0.000435
2,FEDFUNDS,Engineered + Base Features,0.000477,0.021840,0.000442,0.020998
0,FEDFUNDS,Base Features,0.000482,0.021964,0.000454,0.021305
3,FEDFUNDS,Lagged + Base + Engineered Features,0.000753,0.027438,0.000753,0.027436
4,FEDFUNDS,Top_Lagged + Base Features,0.000753,0.027438,0.000753,0.027436
1,FEDFUNDS,Engineered Features,0.000797,0.028230,0.000795,0.028201
5,FEDFUNDS,Top_Lagged + Engineered Features,0.000875,0.029587,0.000814,0.028504


RESULTS FOR TARGET: GDP


,target,combination,xgboost_mse,xgboost_rmse,aggregated_mse,aggregated_rmse
2,GDP,Engineered + Base Features,0.000378,0.019433,0.000354,0.018812
6,GDP,Top_Lagged + Engineered + Base Features,0.000382,0.019536,0.000359,0.000359
0,GDP,Base Features,0.000482,0.021964,0.000454,0.021305
3,GDP,Lagged + Base + Engineered Features,0.000753,0.027438,0.000753,0.027436
4,GDP,Top_Lagged + Base Features,0.000753,0.027438,0.000753,0.027436
5,GDP,Top_Lagged + Engineered Features,0.001087,0.032964,0.001161,0.034056
1,GDP,Engineered Features,0.001356,0.036826,0.001547,0.039259


...

,target,combination,xgboost_mse,xgboost_rmse,aggregated_mse,aggregated_rmse
6,HOUST,Top_Lagged + Engineered + Base Features,0.000430,0.020733,0.000416,0.000416
2,HOUST,Engineered + Base Features,0.000425,0.020626,0.000395,0.019857
0,HOUST,Base Features,0.000482,0.021964,0.000454,0.021305
3,HOUST,Lagged + Base + Engineered Features,0.000753,0.027438,0.000753,0.027436
4,HOUST,Top_Lagged + Base Features,0.000753,0.027438,0.000753,0.027436
5,HOUST,Top_Lagged + Engineered Features,0.001092,0.033047,0.000956,0.030833
1,HOUST,Engineered Features,0.001345,0.036670,0.001395,0.037338


RESULTS FOR TARGET: DSPI


,target,combination,xgboost_mse,xgboost_rmse,aggregated_mse,aggregated_rmse
0,DSPI,Base Features,0.000482,0.021964,0.000454,0.021305
2,DSPI,Engineered + Base Features,0.000498,0.022307,0.000466,0.021577
6,DSPI,Top_Lagged + Engineered + Base Features,0.000484,0.021990,0.000486,0.000486
3,DSPI,Lagged + Base + Engineered Features,0.000753,0.027438,0.000753,0.027436
4,DSPI,Top_Lagged + Base Features,0.000753,0.027438,0.000753,0.027436
5,DSPI,Top_Lagged + Engineered Features,0.001052,0.032435,0.001012,0.031810
1,DSPI,Engineered Features,0.001056,0.032488,0.001167,0.034124


,target,combination,xgboost_mse,xgboost_rmse,aggregated_mse,aggregated_rmse
6,DSPI,Top_Lagged + Engineered + Base Features,0.000484,0.021990,0.000486,0.000486
0,DSPI,Base Features,0.000482,0.021964,0.000454,0.021305
2,DSPI,Engineered + Base Features,0.000498,0.022307,0.000466,0.021577
3,DSPI,Lagged + Base + Engineered Features,0.000753,0.027438,0.000753,0.027436
4,DSPI,Top_Lagged + Base Features,0.000753,0.027438,0.000753,0.027436
5,DSPI,Top_Lagged + Engineered Features,0.001052,0.032435,0.001012,0.031810
1,DSPI,Engineered Features,0.001056,0.032488,0.001167,0.034124


XGBoostError: [14:05:41] C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0b3782d1791676daf-1\xgboost\xgboost-ci-windows\src\data\iterative_dmatrix.cc:202: Check failed: n_features >= 1 (0 vs. 1) : Data must has at least 1 column.

In [236]:
feature_space = pd.read_csv('feature_engineering_results_round2.csv')
top_lagged_features = pd.read_csv('top_lagged_feature_results.csv')

target = 'FEDFUNDS'
feature_space_mask = (feature_space==target).any(axis=1)
feature_space_list = feature_space[feature_space_mask]['final_feature_space'].values.tolist()

feature_space_series =pd.Series(feature_space_list)
feature_space_series

# Example where feature_space is a pandas Series that contains string representations of lists
# Convert the string representations to actual lists
feature_space = feature_space_series.apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Ensure both feature_space and top_lagged_features are lists
feature_space_list = feature_space.tolist()  # May contain lists as elements
# top_lagged_features_list = top_lagged_features.tolist()
feature_space_list
# # Flatten the feature_space_list to remove any nested lists
flat_feature_space = [item for sublist in feature_space_list for item in sublist] if any(isinstance(i, list) for i in feature_space_list) else feature_space_list
top_lagged_features_rows
top_lagged_features_list = []
for feature in top_lagged_features_rows['feature']:
    top_lagged_features_list.append(feature)

top_lagged_features_list
# Combine the flattened lists
features_to_add = flat_feature_space + top_lagged_features_list

# Display the result

In [221]:
top_lagged_features_rows = top_lagged_features[(top_lagged_features == target).any(axis=1)]

In [229]:
features_to_add

['UNRATE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2',
 'FEDFUNDS_lag_3',
 'FEDFUNDS_lag_6',
 'FEDFUNDS_lag_9']

In [212]:
data = flat_feature_space

if isinstance(data, list):
    print("It's a list")
elif isinstance(data, pd.Series):
    print("It's a pandas Series")
elif isinstance(data, np.ndarray):
    print("It's a numpy array")


It's a list


In [116]:
features_to_add_dict

{'FEDFUNDS': ['FEDFUNDS_lag_3',
  'FEDFUNDS_lag_6',
  'FEDFUNDS_lag_9',
  '[\'UNRATE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2\']'],
 'GDP': ['GDP_lag_1',
  'GDP_lag_4',
  'GDP_lag_3',
  '[\'CUMFNS__change_quantiles__f_agg_"var"__isabs_True__qh_0.8__ql_0.2\', \'CPIAUCSL__change_quantiles__f_agg_"mean"__isabs_True__qh_0.8__ql_0.4\']'],
 'CPIAUCSL': ['CPIAUCSL_lag_3',
  'CPIAUCSL_lag_5',
  'CPIAUCSL_lag_4',
  '[\'PCEPI__fft_coefficient__attr_"angle"__coeff_0\', \'CUSR0000SAH1__large_standard_deviation__r_0.25\', \'CUMFNS__large_standard_deviation__r_0.30000000000000004\', \'CPILFESL__large_standard_deviation__r_0.30000000000000004\', \'DGS10__longest_strike_below_mean\', \'HOUST__number_peaks__n_50\', \'CUMFNS__has_duplicate_min\', \'BAA__sum_of_reoccurring_values\', \'CPIAUCSL__large_standard_deviation__r_0.4\']'],
 'CUSR0000SAH1': ['CUSR0000SAH1_lag_4',
  'CUSR0000SAH1_lag_9',
  'CUSR0000SAH1_lag_3',
  '[\'PRFI__fft_coefficient__attr_"angle"__coeff_0\', \'BAA__symmetr

In [114]:
for value in combined:
    print(value)    

USREC_lag_2
USREC_lag_3
USREC_lag_4
['FEDFUNDS__variance_larger_than_standard_deviation', 'FEDFUNDS__has_duplicate_max', 'FEDFUNDS__has_duplicate_min']
